In [1]:
!pip install folktables ucimlrepo

In [2]:
# """
# AGAD v4-HC-fix3  —  All fixes from fix2 review
# ================================================

# FIX 1 — ACS Employment EO: epoch-level cumulative TPR tracking
# ---------------------------------------------------------------
# Problem (fix2):
#   Per-batch DP-fallback fires almost always for sparse intersectional
#   subgroups (disabled & age_old) on a 4096 batch because pos_sg is
#   nearly always < MIN_POS_FOR_EO=4. So AGAD-EO was silently running
#   DP-style sg_loss the whole time, yet the EO audit still penalised it —
#   training signal and eval signal were decoupled.

# Fix applied:
#   Maintain a running EO accumulator (epoch_pos_sg, epoch_prob_sg,
#   epoch_pos_all, epoch_prob_all) across all batches. At the end of each
#   epoch, compute the *epoch-level TPR gap* for every top-k subgroup mask
#   and inject it as an additive penalty into the next epoch's sg_loss
#   scale. This way sparse subgroups accumulate enough positives over the
#   full epoch to produce a meaningful gradient signal, exactly like the
#   validation audit does.

#   Concretely:
#     - During the batch loop, accumulate tensors per top-k mask.
#     - After the epoch loop, compute epoch_eo_penalties dict[mask_idx] -> float.
#     - In the next epoch's batch sg_loss, multiply sg_loss contribution
#       by (1 + epoch_eo_penalties[i]) to upweight masks that had large
#       epoch-level EO gaps. This keeps gradients flowing per-batch while
#       correcting for the sparse-positive problem at epoch resolution.
#   The DP-fallback is kept but MIN_POS_FOR_EO is lowered to 2 since the
#   epoch-level reweighting now handles the signal amplification.

# FIX 2 — Proper differentiable AUC proxy for pareto_w
# ------------------------------------------------------
# Problem (fix2):
#   1 - prob.mean() is mean predicted probability, not AUC. It pushes the
#   model to predict higher probs overall (accuracy/calibration nudge) not
#   to trade off fairness vs discriminability. On Employment EO, higher
#   pareto_w made fairness *worse* (0.4564 → 0.4783), the opposite of intent.

# Fix applied:
#   Replace with a differentiable AUC proxy based on the Wilcoxon-Mann-
#   Whitney statistic approximation:
#     auc_proxy = mean_{i in pos, j in neg} sigmoid(s*(prob_i - prob_j))
#   where s=10 is a sharpness parameter. This is the standard
#   "optimising AUC directly" approach. Gradients push positive-class
#   predictions above negative-class ones, which is exactly what AUC
#   measures. Different pareto_w values now meaningfully trade off
#   fairness penalty weight vs discriminability.

#   To keep compute tractable on large batches (4096), we subsample
#   min(64, n_pos) positives and min(64, n_neg) negatives per batch.

# FIX 3 — Remove redundant forward_no_grl
# -----------------------------------------
# Problem: IntersectionAdversaryHead had two near-identical forward methods
# (forward with GRL, forward_no_grl without). The adversary update used
# forward_no_grl on a detached z, which is equivalent to forward with
# alpha=0 on a detached z.

# Fix applied:
#   Remove forward_no_grl. Add alpha parameter to forward() — when
#   called with alpha=0 the GRL becomes identity (no gradient reversal).
#   Adversary update calls adv(z_d, alpha=0); encoder update calls
#   adv(z, alpha=grl_alpha) as before. API is cleaner and logic is
#   identical.

# No other changes from v4-HC-fix2.
# """

# # ════════════════════════════════════════════════════════════════════════════
# #  Imports & global config
# # ════════════════════════════════════════════════════════════════════════════

# import os, gc, json, time, copy, random, warnings, itertools
# import numpy as np
# import pandas as pd
# import torch
# import torch.nn as nn
# import torch.optim as optim
# from torch.utils.data import DataLoader, TensorDataset
# from sklearn.model_selection import train_test_split
# from sklearn.preprocessing import StandardScaler
# from sklearn.metrics import accuracy_score, roc_auc_score
# import matplotlib
# matplotlib.use("Agg")
# import matplotlib.pyplot as plt
# import matplotlib.patches as mpatches
# from matplotlib.gridspec import GridSpec

# warnings.filterwarnings("ignore")

# WORK_DIR = "/kaggle/working"
# PLOT_DIR = os.path.join(WORK_DIR, "agad_plots_hc_fix3")
# os.makedirs(PLOT_DIR, exist_ok=True)

# DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# SEEDS_FINAL    = [42]
# RUN_DATASETS   = ["adult", "acs_income", "acs_employment", "communities_crime"]
# PARETO_SWEEP_W = [0.00, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30]

# HIDDEN_DIM   = 128
# DROPOUT      = 0.25
# LR           = 1e-3
# WEIGHT_DECAY = 1e-4
# PATIENCE     = 35
# GRL_MAX      = 1.0
# ADV_STEPS    = 3
# ADV_LR_MULT  = 2.0
# LAMBDA_MAX   = 2.0
# PHASE1_END   = 10
# PHASE2_END   = 20
# MIN_SG_N     = 40
# MIN_SG_FRAC  = 0.03
# MAX_SG_FRAC  = 0.50
# MAX_DEPTH    = 3
# TOP_K_REPORT = 5

# # FIX 1: lowered from 4 — epoch-level reweighting handles signal now
# MIN_POS_FOR_EO = 2

# # FIX 2: AUC proxy sharpness and subsample size
# AUC_PROXY_SHARPNESS  = 10.0
# AUC_PROXY_SUBSAMPLE  = 64

# FULL_CFG = dict(
#     adult             = dict(epochs=80,  batch=2048),
#     acs_income        = dict(epochs=120, batch=4096),
#     acs_employment    = dict(epochs=140, batch=4096),
#     communities_crime = dict(epochs=130, batch=256),
# )

# VANILLA_ACC = dict(
#     adult=0.7886,
#     acs_income=0.7978,
#     acs_employment=0.8075,
#     communities_crime=0.7118,
# )
# ACC_TOLERANCE = 0.05

# # ════════════════════════════════════════════════════════════════════════════
# #  HARDCODED HYPERPARAMETERS  (unchanged from fix2)
# # ════════════════════════════════════════════════════════════════════════════

# BEST_PARAMS = {
#     "adult": {
#         "eo": dict(gamma_sg=0.42, lambda0=0.598, alpha=14.48, pareto_w=0.07,
#                    task_weight=1.85, top_k_loss=1, sg_warmup=10),
#         "dp": dict(gamma_sg=0.36, lambda0=0.65,  alpha=14.48, pareto_w=0.13,
#                    task_weight=1.85, top_k_loss=1, sg_warmup=10),
#     },
#     "acs_income": {
#         "eo": dict(gamma_sg=0.18, lambda0=0.59, alpha=12.82, pareto_w=0.14,
#                    task_weight=1.40, top_k_loss=2, sg_warmup=10),
#         "dp": dict(gamma_sg=0.62, lambda0=0.55, alpha=7.15,  pareto_w=0.05,
#                    task_weight=1.58, top_k_loss=4, sg_warmup=15),
#     },
#     "acs_employment": {
#         "eo": dict(
#             gamma_sg    = 1.30,
#             lambda0     = 0.35,
#             alpha       = 10.0,
#             pareto_w    = 0.076,
#             task_weight = 1.40,
#             top_k_loss  = 5,
#             sg_warmup   = 30,
#         ),
#         "dp": dict(gamma_sg=0.18, lambda0=0.40, alpha=10.0, pareto_w=0.053,
#                    task_weight=1.55, top_k_loss=1, sg_warmup=15),
#     },
#     "communities_crime": {
#         "eo": dict(gamma_sg=0.35, lambda0=0.49, alpha=12.36, pareto_w=0.04,
#                    task_weight=1.92, top_k_loss=4, sg_warmup=24),
#         "dp": dict(gamma_sg=0.35, lambda0=0.58, alpha=12.36, pareto_w=0.08,
#                    task_weight=1.92, top_k_loss=4, sg_warmup=24),
#     },
# }

# DEFAULT_HPARAMS = dict(gamma_sg=0.20, lambda0=0.30, alpha=8.0, pareto_w=0.12,
#                        task_weight=2.0, top_k_loss=3, sg_warmup=20)

# SGPEN_GAMMA = dict(adult=0.20, acs_income=0.30,
#                    acs_employment=0.50, communities_crime=0.10)

# PALETTE = {
#     "vanilla"       : "#6c757d",
#     "adv_eo"        : "#4e9af1",
#     "adv_dp"        : "#1a6fc4",
#     "sgpen_eo"      : "#f4a261",
#     "sgpen_dp"      : "#e76f51",
#     "agad_eo_tuned" : "#2dc653",
#     "agad_dp_tuned" : "#1a7c34",
# }

# print("=" * 70)
# print("  AGAD v4-HC-fix3  —  All review fixes applied")
# print("=" * 70)
# print(f"  Device   : {DEVICE}")
# print(f"  Datasets : {RUN_DATASETS}")
# print(f"  PATIENCE : {PATIENCE}")
# print(f"  Seeds    : {SEEDS_FINAL}")
# print()
# print("  KEY CHANGES vs v4-HC-fix2:")
# print("    FIX 1 — Epoch-level cumulative TPR for sparse EO subgroups")
# print("    FIX 2 — WMW differentiable AUC proxy (replaces mean-prob proxy)")
# print("    FIX 3 — Removed redundant forward_no_grl; unified forward(alpha)")
# print("=" * 70)


# # ════════════════════════════════════════════════════════════════════════════
# #  Utilities  (unchanged)
# # ════════════════════════════════════════════════════════════════════════════

# def set_seed(s):
#     random.seed(s); np.random.seed(s)
#     torch.manual_seed(s); torch.cuda.manual_seed_all(s)

# def ensure_binary(sv):
#     sv = np.asarray(sv).ravel(); u = np.unique(sv)
#     if len(u) <= 1: return np.zeros(len(sv), int)
#     if len(u) == 2: return (sv == u[1]).astype(int)
#     maj = u[np.argmax([(sv == v).sum() for v in u])]
#     return (sv != maj).astype(int)

# def safe_auc(yt, yp):
#     try:    return float(roc_auc_score(yt, yp)) if len(np.unique(yt)) >= 2 else float("nan")
#     except: return float("nan")

# def strat_split(X, y, sens_list, test_size=0.2, seed=42):
#     key = np.array(y).astype(str)
#     for s in sens_list:
#         key = key + "_" + np.array(s).astype(str)
#     u, c = np.unique(key, return_counts=True)
#     key[np.isin(key, u[c < 2])] = np.array(y)[np.isin(key, u[c < 2])].astype(str)
#     try:
#         return train_test_split(np.arange(len(y)), test_size=test_size,
#                                 stratify=key, random_state=seed)
#     except:
#         return train_test_split(np.arange(len(y)), test_size=test_size,
#                                 stratify=y, random_state=seed)

# def eo_gap(y_true, y_prob, mask):
#     sg_t = y_true[mask]; sg_p = y_prob[mask]
#     ot_t = y_true[~mask]; ot_p = y_prob[~mask]
#     if len(np.unique(sg_t)) < 2 or len(np.unique(ot_t)) < 2: return 0.0
#     tpr_sg = sg_p[sg_t == 1].mean() if (sg_t == 1).sum() > 0 else 0.0
#     tpr_ot = ot_p[ot_t == 1].mean() if (ot_t == 1).sum() > 0 else 0.0
#     fpr_sg = sg_p[sg_t == 0].mean() if (sg_t == 0).sum() > 0 else 0.0
#     fpr_ot = ot_p[ot_t == 0].mean() if (ot_t == 0).sum() > 0 else 0.0
#     return float(max(abs(tpr_sg - tpr_ot), abs(fpr_sg - fpr_ot)))

# def dp_gap(y_prob, mask):
#     return float(abs(y_prob[mask].mean() - y_prob.mean()))

# def fg_metric(y_true, y_prob, sgs, k=TOP_K_REPORT, metric="eo"):
#     gaps = []
#     for sg in sgs:
#         mem = sg["mem"]
#         if len(np.unique(y_true[mem])) < 2: continue
#         g = eo_gap(y_true, y_prob, mem) if metric == "eo" else dp_gap(y_prob, mem)
#         gaps.append(g)
#     if not gaps: return 0.0
#     gaps.sort(reverse=True)
#     return float(np.mean(gaps[:k]))

# def get_depth_limit(epoch):
#     if epoch < PHASE1_END: return 1
#     if epoch < PHASE2_END: return 2
#     return 3

# def enumerate_subgroups(sv_bin_list, attr_names, n, max_depth=2):
#     n_attr = len(attr_names)
#     min_n  = max(MIN_SG_N, int(MIN_SG_FRAC * n))
#     max_n  = int(MAX_SG_FRAC * n)
#     sgs, seen = [], set()
#     for mask in range(1, 2 ** n_attr):
#         active = [i for i in range(n_attr) if mask & (1 << i)]
#         if len(active) > max_depth: continue
#         for vals in range(2 ** len(active)):
#             req = [(vals >> j) & 1 for j in range(len(active))]
#             mem = np.ones(n, dtype=bool)
#             for ai, rv in zip(active, req):
#                 mem &= (sv_bin_list[ai] == rv)
#             key = mem.tobytes()
#             if key in seen: continue
#             seen.add(key)
#             sz = int(mem.sum())
#             if sz < min_n or sz > max_n: continue
#             spec = [(attr_names[i], req[j]) for j, i in enumerate(active)]
#             sgs.append({"mem": mem, "spec": spec})
#     return sgs

# def audit(epoch, y_val, p_val, sv_val_bin, attr_names, metric="eo"):
#     depth  = min(get_depth_limit(epoch), MAX_DEPTH)
#     n      = len(p_val)
#     sgs    = enumerate_subgroups(sv_val_bin, attr_names, n, max_depth=depth)
#     ranked = []
#     for sg in sgs:
#         mem = sg["mem"]
#         if len(np.unique(y_val[mem])) < 2: continue
#         gap = eo_gap(y_val, p_val, mem) if metric == "eo" else dp_gap(p_val, mem)
#         ranked.append((gap, sg["spec"], mem))
#     ranked.sort(key=lambda x: -x[0])
#     worst_gap  = ranked[0][0] if ranked else 0.0
#     worst_spec = ranked[0][1] if ranked else None
#     topk_specs = [r[1] for r in ranked[:6]]
#     fg_k       = float(np.mean([r[0] for r in ranked[:TOP_K_REPORT]])) if ranked else 0.0
#     return float(worst_gap), worst_spec, float(fg_k), topk_specs

# def adaptive_lambda(V_t, lambda0, alpha):
#     return min(lambda0 * (1.0 + alpha * V_t), LAMBDA_MAX)

# def build_intersection_labels(sv_bin_list, n_attrs):
#     n      = len(sv_bin_list[0])
#     labels = np.zeros(n, dtype=np.int64)
#     for sv in sv_bin_list:
#         labels = labels * 2 + np.asarray(sv, dtype=np.int64)
#     return labels

# def spec_to_train_mask(spec, attr_names, sv_tr_np, n_tr):
#     wm = np.ones(n_tr, dtype=bool)
#     for (attr_name, val) in spec:
#         if attr_name in attr_names:
#             ai = attr_names.index(attr_name)
#             wm &= (sv_tr_np[ai] == val)
#     return wm

# def evaluate(proba, y, sv_bin_list, attr_names, tag="", verbose=True):
#     pred  = (proba > 0.5).astype(int)
#     acc   = float(accuracy_score(y, pred))
#     auc   = safe_auc(y, proba)
#     n     = len(proba)
#     sgs   = enumerate_subgroups(sv_bin_list, attr_names, n, max_depth=MAX_DEPTH)
#     wc_eo = max((eo_gap(y, proba, sg["mem"])
#                  for sg in sgs if len(np.unique(y[sg["mem"]])) >= 2), default=0.0)
#     wc_dp = max((dp_gap(proba, sg["mem"]) for sg in sgs), default=0.0)
#     fg_eo = fg_metric(y, proba, sgs, k=TOP_K_REPORT, metric="eo")
#     fg_dp = fg_metric(y, proba, sgs, k=TOP_K_REPORT, metric="dp")
#     if verbose:
#         print(f"    [{tag:<32}]  WC-EO={wc_eo:.4f}  WC-DP={wc_dp:.4f}  "
#               f"FG-EO={fg_eo:.4f}  FG-DP={fg_dp:.4f}  Acc={acc:.4f}  AUC={auc:.4f}")
#     return {"wc_eo": wc_eo, "wc_dp": wc_dp, "fg_eo": fg_eo,
#             "fg_dp": fg_dp, "acc": acc, "auc": auc}

# def aggregate_seeds(results_list):
#     keys = ["wc_eo", "wc_dp", "fg_eo", "fg_dp", "acc", "auc"]
#     agg  = {}
#     for k in keys:
#         vals = [r[k] for r in results_list if k in r and r[k] is not None]
#         agg[k]          = float(np.mean(vals)) if vals else float("nan")
#         agg[k + "_std"] = float(np.std(vals))  if len(vals) > 1 else 0.0
#         agg[k + "_all"] = [float(v) for v in vals]
#     return agg

# def print_section(title, width=70):
#     print(); print("=" * width); print(f"  {title}"); print("=" * width)

# def print_subsection(title, width=70):
#     print(); print("-" * width); print(f"  {title}"); print("-" * width)


# # ════════════════════════════════════════════════════════════════════════════
# #  FIX 2: Differentiable WMW AUC proxy
# # ════════════════════════════════════════════════════════════════════════════

# def wmw_auc_proxy(prob, y_binary, sharpness=AUC_PROXY_SHARPNESS,
#                   n_sub=AUC_PROXY_SUBSAMPLE):
#     """
#     Wilcoxon-Mann-Whitney differentiable AUC proxy.
#     auc ≈ E_{i~pos, j~neg}[ sigmoid(s*(p_i - p_j)) ]
#     Subsamples pos/neg to keep compute O(n_sub^2) not O(n^2).
#     Returns scalar tensor; grad flows through prob.
#     """
#     pos_idx = (y_binary == 1).nonzero(as_tuple=True)[0]
#     neg_idx = (y_binary == 0).nonzero(as_tuple=True)[0]
#     if len(pos_idx) == 0 or len(neg_idx) == 0:
#         # Can't compute AUC — return neutral value, no gradient
#         return torch.tensor(0.5, device=prob.device)
#     # Subsample for efficiency
#     if len(pos_idx) > n_sub:
#         perm = torch.randperm(len(pos_idx), device=prob.device)[:n_sub]
#         pos_idx = pos_idx[perm]
#     if len(neg_idx) > n_sub:
#         perm = torch.randperm(len(neg_idx), device=prob.device)[:n_sub]
#         neg_idx = neg_idx[perm]
#     p_pos = prob[pos_idx]   # (n_pos,)
#     p_neg = prob[neg_idx]   # (n_neg,)
#     # Outer difference: (n_pos, n_neg)
#     diff = p_pos.unsqueeze(1) - p_neg.unsqueeze(0)
#     return torch.sigmoid(sharpness * diff).mean()


# # ════════════════════════════════════════════════════════════════════════════
# #  Model components  — FIX 3: unified forward(alpha)
# # ════════════════════════════════════════════════════════════════════════════

# class GRL(torch.autograd.Function):
#     @staticmethod
#     def forward(ctx, x, alpha):
#         ctx.alpha = alpha; return x.clone()
#     @staticmethod
#     def backward(ctx, g):
#         return -ctx.alpha * g, None

# class GradReversal(nn.Module):
#     def __init__(self, alpha=1.0):
#         super().__init__(); self.alpha = alpha
#     def forward(self, x):
#         return GRL.apply(x, self.alpha)

# class Encoder(nn.Module):
#     def __init__(self, in_dim):
#         super().__init__()
#         h = HIDDEN_DIM
#         self.rep_dim = h // 2 + 32
#         self.net = nn.Sequential(
#             nn.Linear(in_dim, h),         nn.BatchNorm1d(h),            nn.GELU(), nn.Dropout(DROPOUT),
#             nn.Linear(h, 256),            nn.BatchNorm1d(256),          nn.GELU(), nn.Dropout(DROPOUT),
#             nn.Linear(256, 128),          nn.BatchNorm1d(128),          nn.GELU(), nn.Dropout(DROPOUT * 0.5),
#             nn.Linear(128, self.rep_dim), nn.BatchNorm1d(self.rep_dim), nn.GELU(), nn.Dropout(DROPOUT * 0.5),
#         )
#         for m in self.modules():
#             if isinstance(m, nn.Linear):
#                 nn.init.kaiming_normal_(m.weight); nn.init.zeros_(m.bias)
#     def forward(self, x): return self.net(x)

# class PredictorHead(nn.Module):
#     def __init__(self, rep_dim):
#         super().__init__()
#         self.net = nn.Sequential(
#             nn.Linear(rep_dim, rep_dim // 2), nn.GELU(),
#             nn.Linear(rep_dim // 2, 1),
#         )
#     def forward(self, z): return self.net(z).squeeze(1)

# class IntersectionAdversaryHead(nn.Module):
#     """
#     FIX 3: Removed forward_no_grl. Unified forward(z, alpha) where
#     alpha=0 gives identity (no gradient reversal) for adversary updates,
#     alpha=grl_alpha gives full reversal for encoder updates.
#     """
#     def __init__(self, rep_dim, n_marginal_attrs, n_intersection_classes):
#         super().__init__()
#         h = HIDDEN_DIM // 2
#         self.grl   = GradReversal(1.0)
#         self.trunk = nn.Sequential(
#             nn.Linear(rep_dim, h), nn.GELU(), nn.Dropout(DROPOUT),
#             nn.Linear(h, h // 2), nn.GELU(),
#         )
#         self.marginal_out     = nn.Linear(h // 2, n_marginal_attrs)
#         self.intersection_out = nn.Linear(h // 2, n_intersection_classes)
#         for m in self.modules():
#             if isinstance(m, nn.Linear):
#                 nn.init.kaiming_normal_(m.weight); nn.init.zeros_(m.bias)

#     def set_alpha(self, a):
#         self.grl.alpha = a

#     def forward(self, z, alpha=None):
#         """
#         alpha=None  → use self.grl.alpha (gradient reversal, for encoder update)
#         alpha=0     → no gradient reversal (for adversary update on detached z)
#         alpha=float → override grl alpha
#         """
#         if alpha is not None:
#             self.grl.alpha = alpha
#         h = self.trunk(self.grl(z))
#         return self.marginal_out(h), self.intersection_out(h)


# # ════════════════════════════════════════════════════════════════════════════
# #  FIX 1 helper: epoch-level EO accumulator
# # ════════════════════════════════════════════════════════════════════════════

# class EpochEOAccumulator:
#     """
#     Accumulates per-subgroup TPR statistics across all batches in an epoch.
#     After the epoch, compute_penalties() returns a dict mask_idx -> float
#     representing the epoch-level EO gap for each top-k subgroup mask.
#     These are used to reweight sg_loss contributions next epoch, ensuring
#     sparse intersectional subgroups (low per-batch positive count) still
#     drive gradients when aggregated at epoch resolution.
#     """
#     def __init__(self, n_masks):
#         self.n_masks = n_masks
#         self.reset()

#     def reset(self):
#         # Accumulators: sum of probabilities and counts for pos/neg per mask
#         self.sg_pos_prob_sum  = [0.0] * self.n_masks
#         self.sg_pos_count     = [0]   * self.n_masks
#         self.all_pos_prob_sum = 0.0
#         self.all_pos_count    = 0

#     def update(self, prob_np, y_np, batch_sg_masks_np):
#         """
#         prob_np, y_np: numpy arrays for the batch
#         batch_sg_masks_np: list of bool arrays, one per top-k mask
#         """
#         pos_mask = (y_np == 1)
#         self.all_pos_prob_sum += float(prob_np[pos_mask].sum()) if pos_mask.any() else 0.0
#         self.all_pos_count    += int(pos_mask.sum())
#         for i, sg_mask in enumerate(batch_sg_masks_np):
#             sg_pos = sg_mask & pos_mask
#             self.sg_pos_prob_sum[i] += float(prob_np[sg_pos].sum()) if sg_pos.any() else 0.0
#             self.sg_pos_count[i]    += int(sg_pos.sum())

#     def compute_penalties(self):
#         """Returns dict i -> epoch-level EO gap for mask i."""
#         if self.all_pos_count == 0:
#             return {i: 0.0 for i in range(self.n_masks)}
#         tpr_all = self.all_pos_prob_sum / max(self.all_pos_count, 1)
#         penalties = {}
#         for i in range(self.n_masks):
#             if self.sg_pos_count[i] == 0:
#                 penalties[i] = 0.0
#             else:
#                 tpr_sg = self.sg_pos_prob_sum[i] / self.sg_pos_count[i]
#                 penalties[i] = abs(tpr_sg - tpr_all)
#         return penalties


# # ════════════════════════════════════════════════════════════════════════════
# #  AGAD training  — PATCHED with FIX 1 + FIX 2 + FIX 3
# # ════════════════════════════════════════════════════════════════════════════

# def run_agad_with_hparams(d, cfg, metric, hp, seed=42,
#                            verbose=False, epoch_verbose=False):
#     set_seed(seed)
#     gamma_sg    = hp["gamma_sg"]
#     lambda0     = hp["lambda0"]
#     alpha_val   = hp["alpha"]
#     pareto_w    = hp["pareto_w"]
#     task_weight = hp["task_weight"]
#     top_k_loss  = hp["top_k_loss"]
#     sg_warmup   = hp["sg_warmup"]

#     n_attrs = len(d["attr_names"])
#     n_inter = 2 ** n_attrs

#     Xt   = torch.tensor(d["X_tr"],  dtype=torch.float32, device=DEVICE)
#     yt   = torch.tensor(d["y_tr"],  dtype=torch.float32, device=DEVICE)
#     Xv   = torch.tensor(d["X_val"], dtype=torch.float32, device=DEVICE)
#     Xte  = torch.tensor(d["X_te"],  dtype=torch.float32, device=DEVICE)
#     sv_t = [torch.tensor(sv, dtype=torch.float32, device=DEVICE) for sv in d["sv_tr"]]
#     inter_labels_np = build_intersection_labels(d["sv_tr"], n_attrs)
#     inter_t  = torch.tensor(inter_labels_np, dtype=torch.long, device=DEVICE)
#     idx_t    = torch.arange(len(d["y_tr"]), dtype=torch.long)
#     sv_tr_np = [np.asarray(sv) for sv in d["sv_tr"]]
#     n_tr     = len(d["y_tr"])

#     enc  = Encoder(d["X_tr"].shape[1]).to(DEVICE)
#     head = PredictorHead(enc.rep_dim).to(DEVICE)
#     adv  = IntersectionAdversaryHead(enc.rep_dim, n_attrs, n_inter).to(DEVICE)
#     bce  = nn.BCEWithLogitsLoss()
#     ce   = nn.CrossEntropyLoss()
#     eps  = torch.tensor(1e-4, device=DEVICE)

#     opt_enc = optim.AdamW(
#         list(enc.parameters()) + list(head.parameters()),
#         lr=LR, weight_decay=WEIGHT_DECAY)
#     opt_adv = optim.AdamW(adv.parameters(),
#                           lr=LR * ADV_LR_MULT, weight_decay=WEIGHT_DECAY)
#     sched  = optim.lr_scheduler.CosineAnnealingLR(
#         opt_enc, cfg["epochs"], eta_min=LR * 0.01)
#     loader = DataLoader(
#         TensorDataset(Xt, yt, idx_t),
#         batch_size=cfg["batch"], shuffle=True, drop_last=True)

#     best_score    = np.inf
#     best_enc      = best_head = None
#     no_imp        = 0
#     V_t           = 0.0
#     topk_masks_tr = []
#     topk_masks_tr_np = []   # FIX 1: numpy versions for accumulator
#     acc_floor     = VANILLA_ACC[d["ds_key"]] - ACC_TOLERANCE

#     # FIX 1: epoch-level EO accumulator (initialised with 0 masks, resized later)
#     eo_accum      = None
#     epoch_penalties = {}   # mask_idx -> float, updated each epoch

#     for epoch in range(cfg["epochs"]):
#         enc.eval(); head.eval()
#         with torch.no_grad():
#             pv_scan = torch.sigmoid(head(enc(Xv))).cpu().numpy()

#         V_t, worst_spec, fg_k, topk_specs = audit(
#             epoch, d["y_val"], pv_scan, d["sv_val"], d["attr_names"], metric=metric)
#         lam_t = adaptive_lambda(V_t, lambda0, alpha_val)

#         if epoch >= sg_warmup and topk_specs:
#             topk_masks_tr = [
#                 torch.tensor(
#                     spec_to_train_mask(spec, d["attr_names"], sv_tr_np, n_tr),
#                     dtype=torch.bool, device=DEVICE)
#                 for spec in topk_specs[:top_k_loss]
#             ]
#             # FIX 1: keep numpy versions for the accumulator
#             topk_masks_tr_np = [m.cpu().numpy() for m in topk_masks_tr]
#             # Re-initialise accumulator when mask set changes size
#             if eo_accum is None or eo_accum.n_masks != len(topk_masks_tr):
#                 eo_accum = EpochEOAccumulator(len(topk_masks_tr))
#                 epoch_penalties = {i: 0.0 for i in range(len(topk_masks_tr))}
#         else:
#             topk_masks_tr    = []
#             topk_masks_tr_np = []

#         grl_alpha = GRL_MAX * max(0.1, epoch / max(1, cfg["epochs"]))
#         # FIX 3: set grl alpha on adv before encoder update; adversary update uses alpha=0
#         adv.set_alpha(grl_alpha)
#         enc.train(); head.train(); adv.train()

#         # FIX 1: reset accumulator at start of each epoch
#         if eo_accum is not None:
#             eo_accum.reset()

#         for xb, yb, bidx in loader:
#             # ── Adversary update — FIX 3: use alpha=0 instead of forward_no_grl ──
#             z_d = enc(xb).detach()
#             for _ in range(ADV_STEPS):
#                 opt_adv.zero_grad(set_to_none=True)
#                 # alpha=0 → GRL acts as identity, no gradient reversal
#                 m_out, i_out = adv(z_d, alpha=0)
#                 loss_a = (
#                     sum(nn.functional.binary_cross_entropy_with_logits(
#                         m_out[:, i], sv_t[i][bidx])
#                         for i in range(n_attrs)) / n_attrs
#                     + ce(i_out, inter_t[bidx]))
#                 loss_a.backward()
#                 nn.utils.clip_grad_norm_(adv.parameters(), 1.0)
#                 opt_adv.step()

#             # ── Encoder + head update ────────────────────────────────────
#             opt_enc.zero_grad(set_to_none=True)
#             z      = enc(xb)
#             logits = head(z)
#             prob   = torch.sigmoid(logits)
#             task_loss = bce(logits, yb)

#             # FIX 2: proper WMW differentiable AUC proxy
#             if pareto_w > 0:
#                 auc_proxy = wmw_auc_proxy(prob, yb.long())
#                 # We want to MAXIMISE AUC, so loss contribution is (1 - auc_proxy)
#                 auc_proxy_loss = 1.0 - auc_proxy
#             else:
#                 auc_proxy_loss = torch.tensor(0.0, device=DEVICE)

#             fair_loss = torch.tensor(0.0, device=DEVICE)
#             # FIX 3: use unified adv.forward with current grl_alpha
#             m_adv, i_adv = adv(z)
#             for i, sv in enumerate(sv_t):
#                 tgt = sv[bidx]
#                 if metric == "eo":
#                     sf = tgt.float(); yf = yb.float()
#                     g0y1 = (1 - sf) * yf;     g1y1 = sf * yf
#                     g0y0 = (1 - sf) * (1 - yf); g1y0 = sf * (1 - yf)
#                     if all(g.sum() > 1e-6 for g in [g0y1, g1y1, g0y0, g1y0]):
#                         tpr0 = (prob * g0y1).sum() / (g0y1.sum() + eps)
#                         tpr1 = (prob * g1y1).sum() / (g1y1.sum() + eps)
#                         fpr0 = (prob * g0y0).sum() / (g0y0.sum() + eps)
#                         fpr1 = (prob * g1y0).sum() / (g1y0.sum() + eps)
#                         fair_loss += torch.max(
#                             torch.abs(tpr0 - tpr1), torch.abs(fpr0 - fpr1))
#                 else:
#                     n0 = (1 - tgt).sum() + eps; n1 = tgt.sum() + eps
#                     fair_loss += torch.abs(
#                         (prob * (1 - tgt)).sum() / n0
#                         - (prob * tgt).sum() / n1)
#                 fair_loss += nn.functional.binary_cross_entropy_with_logits(
#                     m_adv[:, i], tgt)
#             fair_loss += ce(i_adv, inter_t[bidx])

#             # ── Subgroup penalty — FIX 1: epoch-level reweighting ────────
#             sg_loss = torch.tensor(0.0, device=DEVICE)
#             if topk_masks_tr:
#                 n_active = 0
#                 batch_probs_np = prob.detach().cpu().numpy()
#                 batch_y_np     = yb.cpu().numpy()
#                 # Update epoch accumulator (numpy, no grad)
#                 if eo_accum is not None:
#                     batch_sg_masks_np_batch = [
#                         m[bidx.cpu().numpy()] for m in topk_masks_tr_np
#                     ]
#                     eo_accum.update(batch_probs_np, batch_y_np,
#                                     batch_sg_masks_np_batch)

#                 for i, wm_tr in enumerate(topk_masks_tr):
#                     batch_sg_mask = wm_tr[bidx]
#                     if batch_sg_mask.sum() < 2: continue

#                     # FIX 1: epoch-level penalty multiplier
#                     # epoch_penalties[i] is the EO gap computed last epoch;
#                     # multiplier > 1 for subgroups with large past gaps
#                     ep_mult = 1.0 + epoch_penalties.get(i, 0.0)

#                     if metric == "eo":
#                         pos_sg  = batch_sg_mask & (yb == 1)
#                         pos_all = (yb == 1)
#                         if pos_sg.sum() >= MIN_POS_FOR_EO and \
#                            pos_all.sum() > pos_sg.sum():
#                             contrib = torch.abs(
#                                 prob[pos_sg].mean() - prob[pos_all].mean())
#                         else:
#                             # DP-style fallback, but now amplified by epoch penalty
#                             contrib = torch.abs(
#                                 prob[batch_sg_mask].mean() - prob.mean())
#                         sg_loss  += ep_mult * contrib
#                         n_active += 1
#                     else:
#                         contrib  = torch.abs(
#                             prob[batch_sg_mask].mean() - prob.mean())
#                         sg_loss  += ep_mult * contrib
#                         n_active += 1

#                 if n_active > 0:
#                     sg_loss = sg_loss / n_active

#             # FIX 2: pareto_w uses WMW AUC proxy
#             loss = (task_weight * task_loss
#                     + lam_t * fair_loss / n_attrs
#                     + gamma_sg * sg_loss
#                     + pareto_w * auc_proxy_loss)

#             loss.backward()
#             nn.utils.clip_grad_norm_(
#                 list(enc.parameters()) + list(head.parameters()), 0.5)
#             opt_enc.step()

#         # FIX 1: update epoch-level penalties after all batches are done
#         if eo_accum is not None and metric == "eo":
#             epoch_penalties = eo_accum.compute_penalties()

#         sched.step()

#         enc.eval(); head.eval()
#         with torch.no_grad():
#             pv  = torch.sigmoid(head(enc(Xv))).cpu().numpy()
#         acc = accuracy_score(d["y_val"], (pv > 0.5).astype(int))
#         auc = safe_auc(d["y_val"], pv)

#         # Checkpoint score (unchanged formula)
#         score = (V_t
#                  + pareto_w * (1 - auc if not np.isnan(auc) else 0)
#                  + 0.5 * max(0.0, acc_floor - acc))

#         if score < best_score:
#             best_score = score
#             best_enc   = copy.deepcopy(enc.state_dict())
#             best_head  = copy.deepcopy(head.state_dict())
#             no_imp     = 0
#         else:
#             no_imp += 1

#         if epoch_verbose and (epoch % 10 == 0 or epoch == cfg["epochs"] - 1):
#             print(f"      epoch {epoch:03d}  V_t={V_t:.4f}  "
#                   f"lam={lam_t:.3f}  acc={acc:.4f}  auc={auc:.4f}  "
#                   f"score={score:.5f}  no_imp={no_imp}")

#         if no_imp >= PATIENCE:
#             if epoch_verbose:
#                 print(f"      [early stop at epoch {epoch}]")
#             break

#     enc.load_state_dict(best_enc)
#     head.load_state_dict(best_head)
#     enc.eval(); head.eval()
#     with torch.no_grad():
#         proba = torch.sigmoid(head(enc(Xte))).cpu().numpy()

#     tag = f"AGAD-{metric.upper()} (HC)"
#     return evaluate(proba, d["y_te"], d["sv_te"], d["attr_names"],
#                     tag=tag, verbose=verbose)


# # ════════════════════════════════════════════════════════════════════════════
# #  Baselines  — FIX 3: _adv_static uses unified adv.forward(alpha)
# # ════════════════════════════════════════════════════════════════════════════

# def run_vanilla(d, cfg, seed=42):
#     set_seed(seed)
#     Xt  = torch.tensor(d["X_tr"],  dtype=torch.float32, device=DEVICE)
#     yt  = torch.tensor(d["y_tr"],  dtype=torch.float32, device=DEVICE)
#     Xv  = torch.tensor(d["X_val"], dtype=torch.float32, device=DEVICE)
#     Xte = torch.tensor(d["X_te"],  dtype=torch.float32, device=DEVICE)
#     enc  = Encoder(d["X_tr"].shape[1]).to(DEVICE)
#     head = PredictorHead(enc.rep_dim).to(DEVICE)
#     bce  = nn.BCEWithLogitsLoss()
#     opt  = optim.AdamW(list(enc.parameters()) + list(head.parameters()),
#                        lr=LR, weight_decay=WEIGHT_DECAY)
#     sched  = optim.lr_scheduler.CosineAnnealingLR(opt, cfg["epochs"], eta_min=LR * 0.01)
#     loader = DataLoader(TensorDataset(Xt, yt),
#                         batch_size=cfg["batch"], shuffle=True, drop_last=True)
#     best_score = np.inf; best_enc = best_head = None; no_imp = 0
#     for epoch in range(cfg["epochs"]):
#         enc.train(); head.train()
#         for xb, yb in loader:
#             opt.zero_grad(set_to_none=True)
#             bce(head(enc(xb)), yb).backward()
#             opt.step()
#         sched.step()
#         enc.eval(); head.eval()
#         with torch.no_grad():
#             pv = torch.sigmoid(head(enc(Xv))).cpu().numpy()
#         auc = safe_auc(d["y_val"], pv)
#         V, _, _, _ = audit(epoch, d["y_val"], pv, d["sv_val"], d["attr_names"])
#         score = V + 0.12 * (1 - auc if not np.isnan(auc) else 0)
#         if score < best_score:
#             best_score = score; best_enc = copy.deepcopy(enc.state_dict())
#             best_head  = copy.deepcopy(head.state_dict()); no_imp = 0
#         else:
#             no_imp += 1
#         if no_imp >= PATIENCE: break
#     enc.load_state_dict(best_enc); head.load_state_dict(best_head)
#     enc.eval(); head.eval()
#     with torch.no_grad():
#         proba = torch.sigmoid(head(enc(Xte))).cpu().numpy()
#     return evaluate(proba, d["y_te"], d["sv_te"], d["attr_names"], tag="Vanilla NN")


# def _adv_static(d, cfg, metric, gamma_sg_static, seed=42):
#     set_seed(seed)
#     n_attrs = len(d["attr_names"])
#     n_inter = 2 ** n_attrs
#     Xt   = torch.tensor(d["X_tr"],  dtype=torch.float32, device=DEVICE)
#     yt   = torch.tensor(d["y_tr"],  dtype=torch.float32, device=DEVICE)
#     Xv   = torch.tensor(d["X_val"], dtype=torch.float32, device=DEVICE)
#     Xte  = torch.tensor(d["X_te"],  dtype=torch.float32, device=DEVICE)
#     sv_t = [torch.tensor(sv, dtype=torch.float32, device=DEVICE) for sv in d["sv_tr"]]
#     inter_labels_np = build_intersection_labels(d["sv_tr"], n_attrs)
#     inter_t = torch.tensor(inter_labels_np, dtype=torch.long, device=DEVICE)
#     idx_t   = torch.arange(len(d["y_tr"]), dtype=torch.long)
#     enc  = Encoder(d["X_tr"].shape[1]).to(DEVICE)
#     head = PredictorHead(enc.rep_dim).to(DEVICE)
#     adv  = IntersectionAdversaryHead(enc.rep_dim, n_attrs, n_inter).to(DEVICE)
#     bce  = nn.BCEWithLogitsLoss(); ce = nn.CrossEntropyLoss()
#     eps  = torch.tensor(1e-4, device=DEVICE)
#     opt_enc = optim.AdamW(list(enc.parameters()) + list(head.parameters()),
#                           lr=LR, weight_decay=WEIGHT_DECAY)
#     opt_adv = optim.AdamW(adv.parameters(), lr=LR * ADV_LR_MULT, weight_decay=WEIGHT_DECAY)
#     sched  = optim.lr_scheduler.CosineAnnealingLR(opt_enc, cfg["epochs"], eta_min=LR * 0.01)
#     loader = DataLoader(TensorDataset(Xt, yt, idx_t),
#                         batch_size=cfg["batch"], shuffle=True, drop_last=True)
#     best_score = np.inf; best_enc = best_head = None; no_imp = 0
#     for epoch in range(cfg["epochs"]):
#         alpha = GRL_MAX * max(0.1, epoch / max(1, cfg["epochs"]))
#         adv.set_alpha(alpha); enc.train(); head.train(); adv.train()
#         for xb, yb, bidx in loader:
#             z_d = enc(xb).detach()
#             for _ in range(ADV_STEPS):
#                 opt_adv.zero_grad(set_to_none=True)
#                 # FIX 3: alpha=0 instead of forward_no_grl
#                 m_out, i_out = adv(z_d, alpha=0)
#                 la = (sum(nn.functional.binary_cross_entropy_with_logits(
#                           m_out[:, i], sv_t[i][bidx]) for i in range(n_attrs)) / n_attrs
#                       + ce(i_out, inter_t[bidx]))
#                 la.backward(); nn.utils.clip_grad_norm_(adv.parameters(), 1.0); opt_adv.step()
#             opt_enc.zero_grad(set_to_none=True)
#             z = enc(xb); logits = head(z); prob = torch.sigmoid(logits)
#             task_loss = bce(logits, yb)
#             fair_loss = torch.tensor(0.0, device=DEVICE)
#             # FIX 3: use unified forward (grl_alpha already set via set_alpha)
#             m_adv, i_adv = adv(z)
#             for i, sv in enumerate(sv_t):
#                 tgt = sv[bidx]
#                 if metric == "eo":
#                     sf = tgt.float(); yf = yb.float()
#                     g0y1=(1-sf)*yf; g1y1=sf*yf; g0y0=(1-sf)*(1-yf); g1y0=sf*(1-yf)
#                     if all(g.sum() > 1e-6 for g in [g0y1, g1y1, g0y0, g1y0]):
#                         tpr0=(prob*g0y1).sum()/(g0y1.sum()+eps); tpr1=(prob*g1y1).sum()/(g1y1.sum()+eps)
#                         fpr0=(prob*g0y0).sum()/(g0y0.sum()+eps); fpr1=(prob*g1y0).sum()/(g1y0.sum()+eps)
#                         fair_loss += torch.max(torch.abs(tpr0-tpr1), torch.abs(fpr0-fpr1))
#                 else:
#                     n0=(1-tgt).sum()+eps; n1=tgt.sum()+eps
#                     fair_loss += torch.abs((prob*(1-tgt)).sum()/n0 - (prob*tgt).sum()/n1)
#                 fair_loss += nn.functional.binary_cross_entropy_with_logits(m_adv[:, i], tgt)
#             fair_loss += ce(i_adv, inter_t[bidx])
#             sg_loss = torch.tensor(0.0, device=DEVICE)
#             if gamma_sg_static > 0:
#                 gm = prob.mean()
#                 for cls in range(n_inter):
#                     cm = (inter_t[bidx] == cls)
#                     if cm.sum() > 1:
#                         sg_loss += torch.abs(prob[cm].mean() - gm)
#                 sg_loss = sg_loss / max(n_inter, 1)
#             loss = (2.0 * task_loss + 0.3 * fair_loss / n_attrs + gamma_sg_static * sg_loss)
#             loss.backward()
#             nn.utils.clip_grad_norm_(list(enc.parameters()) + list(head.parameters()), 0.5)
#             opt_enc.step()
#         sched.step()
#         enc.eval(); head.eval()
#         with torch.no_grad():
#             pv = torch.sigmoid(head(enc(Xv))).cpu().numpy()
#         auc = safe_auc(d["y_val"], pv)
#         V, _, _, _ = audit(epoch, d["y_val"], pv, d["sv_val"], d["attr_names"], metric=metric)
#         score = V + 0.12 * (1 - auc if not np.isnan(auc) else 0)
#         if score < best_score:
#             best_score = score; best_enc = copy.deepcopy(enc.state_dict())
#             best_head  = copy.deepcopy(head.state_dict()); no_imp = 0
#         else:
#             no_imp += 1
#         if no_imp >= PATIENCE: break
#     enc.load_state_dict(best_enc); head.load_state_dict(best_head)
#     enc.eval(); head.eval()
#     with torch.no_grad():
#         proba = torch.sigmoid(head(enc(Xte))).cpu().numpy()
#     tag = (f"Adv-{metric.upper()} (static)" if gamma_sg_static == 0
#            else f"SubgroupPenalty-{metric.upper()}")
#     return evaluate(proba, d["y_te"], d["sv_te"], d["attr_names"], tag=tag)


# # ════════════════════════════════════════════════════════════════════════════
# #  Data loaders  (unchanged)
# # ════════════════════════════════════════════════════════════════════════════

# def load_adult():
#     from ucimlrepo import fetch_ucirepo
#     repo  = fetch_ucirepo(id=2)
#     X_df  = repo.data.features.copy()
#     y_ser = repo.data.targets.copy()
#     y_raw = y_ser.iloc[:, 0].astype(str).str.strip().str.rstrip(".")
#     y     = (y_raw == ">50K").astype(int).values
#     race_Black = (X_df["race"].astype(str).str.strip() == "Black").astype(int).values
#     race_White = (X_df["race"].astype(str).str.strip() == "White").astype(int).values
#     sex_Female = (X_df["sex"].astype(str).str.strip() == "Female").astype(int).values
#     age_old    = (pd.to_numeric(X_df["age"], errors="coerce").fillna(0) >= 45).astype(int).values
#     X_df = X_df.drop(columns=["race","sex","age","fnlwgt","education-num"], errors="ignore")
#     X_df = pd.get_dummies(X_df)
#     X    = X_df.values.astype(np.float32)
#     valid = ~np.isnan(X).any(axis=1)
#     X, y  = X[valid], y[valid]
#     race_Black=race_Black[valid]; race_White=race_White[valid]
#     sex_Female=sex_Female[valid]; age_old=age_old[valid]
#     tr, te = strat_split(X, y, [race_Black, sex_Female, age_old])
#     sc = StandardScaler()
#     X_tr = sc.fit_transform(X[tr]); X_te = sc.transform(X[te])
#     attr_names = ["race_Black","race_White","sex_Female","age_old"]
#     sv_tr = [race_Black[tr], race_White[tr], sex_Female[tr], age_old[tr]]
#     sv_te = [race_Black[te], race_White[te], sex_Female[te], age_old[te]]
#     tr2, val = strat_split(X_tr, y[tr], sv_tr)
#     return dict(ds_key="adult",
#         X_tr=X_tr[tr2], y_tr=y[tr][tr2], X_val=X_tr[val], y_val=y[tr][val],
#         X_te=X_te, y_te=y[te],
#         sv_tr=[ensure_binary(sv[tr2]) for sv in sv_tr],
#         sv_val=[ensure_binary(sv[val]) for sv in sv_tr],
#         sv_te=[ensure_binary(sv) for sv in sv_te],
#         attr_names=attr_names)

# def load_acs_income():
#     from folktables import ACSDataSource, ACSIncome
#     ds  = ACSDataSource(survey_year="2018", horizon="1-Year", survey="person")
#     acs = ds.get_data(states=["CA"], download=True)
#     features, label, group = ACSIncome.df_to_numpy(acs)
#     row_mask = (acs["AGEP"].fillna(-1) >= 16)
#     for col in ["PINCP","COW","WKHP"]:
#         if col in acs.columns:
#             if col == "WKHP":
#                 row_mask = row_mask & (pd.to_numeric(acs[col], errors="coerce").fillna(0) >= 1)
#             else:
#                 row_mask = row_mask & acs[col].notna()
#     if "ESR" in acs.columns:
#         row_mask = row_mask & (~acs["ESR"].isin(["b", None]) & acs["ESR"].notna())
#     acs_feature_cols = ["AGEP","COW","SCHL","MAR","OCCP","POBP","RELP","WKHP","SEX","RAC1P"]
#     sex_idx  = acs_feature_cols.index("SEX"); race_idx = acs_feature_cols.index("RAC1P")
#     age_idx  = acs_feature_cols.index("AGEP")
#     sex_col  = features[:, sex_idx]; race_col = features[:, race_idx]; age_col = features[:, age_idx]
#     rW=(race_col==1).astype(int); rB=(race_col==2).astype(int)
#     rA=(race_col==6).astype(int); sex=(sex_col==2).astype(int); age_old=(age_col>=45).astype(int)
#     valid    = ~np.isnan(label.astype(float))
#     features = features[valid].astype(np.float32); label = label[valid].astype(int)
#     rW=rW[valid]; rB=rB[valid]; rA=rA[valid]; sex=sex[valid]; age_old=age_old[valid]
#     tr, te = strat_split(features, label, [rW, rB, sex, age_old])
#     sc = StandardScaler()
#     X_tr = sc.fit_transform(features[tr]); X_te = sc.transform(features[te])
#     sv_tr = [rW[tr], rB[tr], rA[tr], sex[tr], age_old[tr]]
#     sv_te = [rW[te], rB[te], rA[te], sex[te], age_old[te]]
#     attr_names = ["race_White","race_Black","race_Asian","sex_Female","age_old"]
#     tr2, val = strat_split(X_tr, label[tr], sv_tr)
#     return dict(ds_key="acs_income",
#         X_tr=X_tr[tr2], y_tr=label[tr][tr2], X_val=X_tr[val], y_val=label[tr][val],
#         X_te=X_te, y_te=label[te],
#         sv_tr=[ensure_binary(sv[tr2]) for sv in sv_tr],
#         sv_val=[ensure_binary(sv[val]) for sv in sv_tr],
#         sv_te=[ensure_binary(sv) for sv in sv_te],
#         attr_names=attr_names)

# def load_acs_employment():
#     from folktables import ACSDataSource, ACSEmployment
#     ds  = ACSDataSource(survey_year="2018", horizon="1-Year", survey="person")
#     acs = ds.get_data(states=["CA"], download=True)
#     features, label, _ = ACSEmployment.df_to_numpy(acs)
#     acs_emp_cols = ["AGEP","SCHL","MAR","DIS","ESP","CIT","MIG","MIL","ANC",
#                     "NATIVITY","DEAR","DEYE","DREM","SEX","RAC1P"]
#     sex_idx  = acs_emp_cols.index("SEX"); race_idx = acs_emp_cols.index("RAC1P")
#     age_idx  = acs_emp_cols.index("AGEP"); dis_idx  = acs_emp_cols.index("DIS")
#     sex_col  = features[:, sex_idx]; race_col = features[:, race_idx]
#     age_col  = features[:, age_idx]; dis_col  = features[:, dis_idx]
#     RACE_MAP = {1:0, 2:1, 3:3, 4:2, 5:2, 6:3, 7:3, 8:3, 9:3}
#     race = np.array([RACE_MAP.get(int(r), 3) for r in race_col])
#     sex  = (sex_col == 2).astype(int)
#     rW=(race==0).astype(int); rB=(race==1).astype(int); rO=(race==3).astype(int)
#     age_old=(age_col>=45).astype(int); disabled=(dis_col==1).astype(int)
#     valid    = ~np.isnan(label.astype(float))
#     features = features[valid].astype(np.float32); label = label[valid].astype(int)
#     rW=rW[valid]; rB=rB[valid]; rO=rO[valid]; sex=sex[valid]
#     age_old=age_old[valid]; disabled=disabled[valid]
#     tr, te = strat_split(features, label, [rW, rB, sex, age_old, disabled])
#     sc = StandardScaler()
#     X_tr = sc.fit_transform(features[tr]); X_te = sc.transform(features[te])
#     sv_tr = [rW[tr], rB[tr], rO[tr], sex[tr], age_old[tr], disabled[tr]]
#     sv_te = [rW[te], rB[te], rO[te], sex[te], age_old[te], disabled[te]]
#     attr_names = ["race_White","race_Black","race_Other","sex_Female","age_old","disabled"]
#     tr2, val = strat_split(X_tr, label[tr], sv_tr)
#     return dict(ds_key="acs_employment",
#         X_tr=X_tr[tr2], y_tr=label[tr][tr2], X_val=X_tr[val], y_val=label[tr][val],
#         X_te=X_te, y_te=label[te],
#         sv_tr=[ensure_binary(sv[tr2]) for sv in sv_tr],
#         sv_val=[ensure_binary(sv[val]) for sv in sv_tr],
#         sv_te=[ensure_binary(sv) for sv in sv_te],
#         attr_names=attr_names)

# def load_communities_crime():
#     from ucimlrepo import fetch_ucirepo
#     repo = fetch_ucirepo(id=183)
#     X_df = repo.data.features.copy()
#     y_df = repo.data.targets.copy()
#     y_cont = pd.to_numeric(y_df.iloc[:, 0], errors="coerce").values
#     valid  = ~np.isnan(y_cont)
#     y_cont = y_cont[valid]; X_df = X_df[valid].reset_index(drop=True)
#     y = (y_cont > np.median(y_cont)).astype(int)
#     def find_col(df, pats):
#         for p in pats:
#             m = [c for c in df.columns if p.lower() in c.lower()]
#             if m: return pd.to_numeric(df[m[0]], errors="coerce")
#         return None
#     col_b = find_col(X_df, ["racepctblack","pctblack"])
#     col_i = find_col(X_df, ["medIncome","medincome"])
#     def binarise(col, high=True):
#         if col is None: return np.zeros(len(y), int)
#         c = col.fillna(col.median()).values
#         return (c > np.median(c)).astype(int) if high else (c < np.median(c)).astype(int)
#     s_race = binarise(col_b, high=True); s_inc = binarise(col_i, high=False)
#     X_num = X_df.apply(pd.to_numeric, errors="coerce")
#     X_num = X_num.dropna(axis=1, thresh=int(0.7*len(X_num))).fillna(X_num.median())
#     drop_cols = [c for c in X_num.columns if any(p.lower() in c.lower()
#                  for p in ["racepct","racePct","medIncome","ViolentCrimes","PctUnemployed"])]
#     X = X_num.drop(columns=drop_cols, errors="ignore").values.astype(np.float32)
#     tr, te = strat_split(X, y, [s_race, s_inc])
#     sc = StandardScaler()
#     X_tr = sc.fit_transform(X[tr]); X_te = sc.transform(X[te])
#     sv_tr = [s_race[tr], s_inc[tr]]; sv_te = [s_race[te], s_inc[te]]
#     attr_names = ["racial_composition","socioeconomic"]
#     tr2, val = strat_split(X_tr, y[tr], sv_tr)
#     return dict(ds_key="communities_crime",
#         X_tr=X_tr[tr2], y_tr=y[tr][tr2], X_val=X_tr[val], y_val=y[tr][val],
#         X_te=X_te, y_te=y[te],
#         sv_tr=[ensure_binary(sv[tr2]) for sv in sv_tr],
#         sv_val=[ensure_binary(sv[val]) for sv in sv_tr],
#         sv_te=[ensure_binary(sv) for sv in sv_te],
#         attr_names=attr_names)

# LOADERS = {
#     "adult"            : load_adult,
#     "acs_income"       : load_acs_income,
#     "acs_employment"   : load_acs_employment,
#     "communities_crime": load_communities_crime,
# }


# # ════════════════════════════════════════════════════════════════════════════
# #  Phase 2 — Sanity check
# # ════════════════════════════════════════════════════════════════════════════

# print_section("PHASE 2 — Sanity check: hardcoded params vs default params")
# print("  (ACS Employment EO highlighted — FIX 1 target)")

# t0 = time.time()

# for ds_key in RUN_DATASETS:
#     print_subsection(f"Dataset: {ds_key.upper()}")
#     cfg       = FULL_CFG[ds_key]
#     d         = LOADERS[ds_key]()
#     acc_floor = VANILLA_ACC[ds_key] - ACC_TOLERANCE

#     for metric in ["eo", "dp"]:
#         hp = BEST_PARAMS[ds_key][metric]
#         print(f"\n  [{metric.upper()}] Hardcoded run ...")
#         r_hc      = run_agad_with_hparams(d, cfg, metric, hp, seed=42, verbose=True)
#         print(f"  [{metric.upper()}] Default run ...")
#         r_default = run_agad_with_hparams(d, cfg, metric, DEFAULT_HPARAMS,
#                                            seed=42, verbose=True)

#         wc_key   = f"wc_{metric}"
#         delta_wc = r_default[wc_key] - r_hc[wc_key]
#         delta_acc= r_hc["acc"] - r_default["acc"]
#         acc_ok   = r_hc["acc"] >= acc_floor
#         decision = "✓ HC WINS" if delta_wc >= 0 else "✗ HC LOSES — review params"

#         print(f"\n  {ds_key} {metric.upper()}:  "
#               f"ΔWC={delta_wc:+.4f}  ΔAcc={delta_acc:+.4f}  "
#               f"acc_ok={acc_ok}  ({decision})")

#     gc.collect()
#     if torch.cuda.is_available(): torch.cuda.empty_cache()


# # ════════════════════════════════════════════════════════════════════════════
# #  Phase 3 — Pareto frontier sweep
# # ════════════════════════════════════════════════════════════════════════════

# print_section("PHASE 3 — Pareto frontier sweep  [WMW AUC proxy]")
# print("  Expect DIFFERENT WC / AUC values at each pareto_w.")

# pareto_curves = {ds: {"eo": [], "dp": []} for ds in RUN_DATASETS}

# for ds_key in RUN_DATASETS:
#     print_subsection(f"Dataset: {ds_key.upper()}")
#     cfg = FULL_CFG[ds_key]
#     d   = LOADERS[ds_key]()

#     for metric in ["eo", "dp"]:
#         bp_base = dict(BEST_PARAMS[ds_key][metric])
#         print(f"\n  [{metric.upper()}] Pareto sweep over pareto_w ...")
#         prev_wc = None
#         for pw in PARETO_SWEEP_W:
#             hp_sweep = dict(bp_base); hp_sweep["pareto_w"] = pw
#             r = run_agad_with_hparams(d, cfg, metric, hp_sweep,
#                                        seed=42, verbose=False)
#             wc = r[f"wc_{metric}"]
#             pareto_curves[ds_key][metric].append({
#                 "pareto_w": pw,
#                 "wc_fairness": wc,
#                 "fg_fairness": r[f"fg_{metric}"],
#                 "acc": r["acc"],
#                 "auc": r["auc"],
#             })
#             changed = "" if prev_wc is None else (
#                 "  ← MOVED" if abs(wc - prev_wc) > 5e-4 else "  (same)")
#             print(f"    pareto_w={pw:.2f}  "
#                   f"WC-{metric.upper()}={wc:.4f}  "
#                   f"AUC={r['auc']:.4f}  Acc={r['acc']:.4f}{changed}")
#             prev_wc = wc

#     gc.collect()
#     if torch.cuda.is_available(): torch.cuda.empty_cache()

# with open(f"{WORK_DIR}/agad_pareto_curves_hc_fix3.json", "w") as f:
#     json.dump(pareto_curves, f, indent=2)
# print(f"\n  Saved → {WORK_DIR}/agad_pareto_curves_hc_fix3.json")


# # ════════════════════════════════════════════════════════════════════════════
# #  Phase 4 — Final evaluation
# # ════════════════════════════════════════════════════════════════════════════

# print_section("PHASE 4 — Final evaluation")

# t1 = time.time()
# all_results = {}

# METHOD_LABELS = {
#     "vanilla"       : "Vanilla NN",
#     "adv_eo"        : "Adv-EO",
#     "adv_dp"        : "Adv-DP",
#     "sgpen_eo"      : "SGPen-EO",
#     "sgpen_dp"      : "SGPen-DP",
#     "agad_eo_tuned" : "AGAD-EO ★",
#     "agad_dp_tuned" : "AGAD-DP ★",
# }
# METHOD_ORDER = list(METHOD_LABELS.keys())

# DS_LABELS = {
#     "adult"            : "Adult Income",
#     "acs_income"       : "ACS Income",
#     "acs_employment"   : "ACS Employment",
#     "communities_crime": "Communities & Crime",
# }

# for ds_key in RUN_DATASETS:
#     print_subsection(f"Dataset: {ds_key.upper()}")
#     cfg = FULL_CFG[ds_key]
#     d   = LOADERS[ds_key]()
#     print(f"  Train={len(d['y_tr'])}  Val={len(d['y_val'])}  Test={len(d['y_te'])}")

#     ds_results = {}

#     for bname, bfn in [
#         ("vanilla",  lambda s: run_vanilla(d, cfg, seed=s)),
#         ("adv_eo",   lambda s: _adv_static(d, cfg, "eo", 0.0, seed=s)),
#         ("adv_dp",   lambda s: _adv_static(d, cfg, "dp", 0.0, seed=s)),
#         ("sgpen_eo", lambda s: _adv_static(d, cfg, "eo", SGPEN_GAMMA[ds_key], seed=s)),
#         ("sgpen_dp", lambda s: _adv_static(d, cfg, "dp", SGPEN_GAMMA[ds_key], seed=s)),
#     ]:
#         print(f"\n  -- {bname} --")
#         seed_results = []
#         for s in SEEDS_FINAL:
#             print(f"    seed {s} ...", end=" ", flush=True)
#             r = bfn(s)
#             seed_results.append(r)
#             print(f"WC-EO={r['wc_eo']:.4f}  WC-DP={r['wc_dp']:.4f}  "
#                   f"Acc={r['acc']:.4f}  AUC={r['auc']:.4f}")
#         ds_results[bname] = aggregate_seeds(seed_results)

#     for metric in ["eo", "dp"]:
#         key = f"agad_{metric}_tuned"
#         hp  = BEST_PARAMS[ds_key][metric]
#         print(f"\n  -- AGAD-{metric.upper()} (HC) --")
#         print(f"     hparams: {hp}")
#         seed_results = []
#         for s in SEEDS_FINAL:
#             print(f"    seed {s} ...")
#             r = run_agad_with_hparams(d, cfg, metric, hp,
#                                        seed=s, verbose=True, epoch_verbose=True)
#             seed_results.append(r)
#         ds_results[key] = aggregate_seeds(seed_results)

#     all_results[ds_key] = ds_results

#     print(f"\n{'─'*108}")
#     print(f"  {'Method':<20}  "
#           f"{'WC-EO':>18}  {'WC-DP':>18}  "
#           f"{'FG-EO':>18}  {'FG-DP':>18}  "
#           f"{'Acc':>14}  {'AUC':>14}")
#     print(f"  {'─'*104}")
#     for name, r in ds_results.items():
#         def fmt(k): return f"{r[k]:.4f}±{r[k+'_std']:.4f}"
#         print(f"  {name:<20}  "
#               f"{fmt('wc_eo'):>18}  {fmt('wc_dp'):>18}  "
#               f"{fmt('fg_eo'):>18}  {fmt('fg_dp'):>18}  "
#               f"{fmt('acc'):>14}  {fmt('auc'):>14}")

#     print(f"\n  AGAD (HC) vs best baseline:")
#     for metric in ["eo", "dp"]:
#         best_base_wc  = min(ds_results[f"adv_{metric}"][f"wc_{metric}"],
#                             ds_results[f"sgpen_{metric}"][f"wc_{metric}"])
#         best_base_fg  = min(ds_results[f"adv_{metric}"][f"fg_{metric}"],
#                             ds_results[f"sgpen_{metric}"][f"fg_{metric}"])
#         best_base_acc = max(ds_results[f"adv_{metric}"]["acc"],
#                             ds_results[f"sgpen_{metric}"]["acc"])
#         key = f"agad_{metric}_tuned"
#         r   = ds_results[key]
#         dwc  = best_base_wc  - r[f"wc_{metric}"]
#         dfg  = best_base_fg  - r[f"fg_{metric}"]
#         dacc = r["acc"]      - best_base_acc
#         acc_ok = r["acc"] >= VANILLA_ACC[ds_key] - ACC_TOLERANCE
#         print(f"    agad_{metric}  "
#               f"ΔWC-{metric.upper()}={dwc:+.5f}  "
#               f"ΔFG-{metric.upper()}={dfg:+.5f}  "
#               f"ΔAcc={dacc:+.5f}  "
#               f"({'✓' if dwc>0 else '✗'}  {'✓' if dfg>0 else '✗'})  "
#               f"acc_ok={'✓' if acc_ok else '✗'}")

#     gc.collect()
#     if torch.cuda.is_available(): torch.cuda.empty_cache()

# phase4_time = time.time() - t1


# # ════════════════════════════════════════════════════════════════════════════
# #  Phase 5 — Graphs  (unchanged)
# # ════════════════════════════════════════════════════════════════════════════

# print_section("PHASE 5 — Generating publication-quality graphs")

# plt.rcParams.update({
#     "font.family": "DejaVu Sans", "font.size": 10,
#     "axes.titlesize": 11, "axes.labelsize": 10,
#     "xtick.labelsize": 8, "ytick.labelsize": 8,
#     "legend.fontsize": 8, "figure.dpi": 150,
#     "axes.spines.top": False, "axes.spines.right": False,
# })

# def fig1_bar_fairness():
#     fig, axes = plt.subplots(2, 4, figsize=(16, 7), sharey=False)
#     fig.suptitle("Figure 1: Worst-Case Fairness Gaps Across Datasets and Methods\n"
#                  "(lower is better; ★ = AGAD HC-fix3)",
#                  fontsize=13, fontweight="bold", y=1.01)
#     for col, ds_key in enumerate(RUN_DATASETS):
#         for row, metric in enumerate(["eo", "dp"]):
#             ax = axes[row][col]; wc_key = f"wc_{metric}"
#             means  = [all_results[ds_key][m][wc_key]          for m in METHOD_ORDER]
#             stds   = [all_results[ds_key][m][wc_key + "_std"] for m in METHOD_ORDER]
#             colors = [PALETTE[m] for m in METHOD_ORDER]
#             x = np.arange(len(METHOD_ORDER))
#             bars = ax.bar(x, means, yerr=stds, color=colors, alpha=0.85,
#                           capsize=3, error_kw={"linewidth": 1.2})
#             ax.set_xticks(x)
#             ax.set_xticklabels([METHOD_LABELS[m] for m in METHOD_ORDER],
#                                rotation=45, ha="right", fontsize=7)
#             ax.set_title(f"{DS_LABELS[ds_key]}\nWC-{'EO' if metric=='eo' else 'DP'}", fontsize=9)
#             ax.set_ylabel("Gap" if col == 0 else "", fontsize=9)
#             ax.yaxis.grid(True, linestyle="--", alpha=0.4); ax.set_axisbelow(True)
#             for i, m in enumerate(METHOD_ORDER):
#                 if "agad" in m:
#                     bars[i].set_edgecolor("black"); bars[i].set_linewidth(1.5)
#     plt.tight_layout()
#     path = f"{PLOT_DIR}/fig1_wc_fairness_bars.png"
#     plt.savefig(path, bbox_inches="tight", dpi=150); plt.close()
#     print(f"  Saved → {path}")

# def fig2_pareto_curves():
#     fig, axes = plt.subplots(2, 4, figsize=(16, 7), sharey=False)
#     fig.suptitle("Figure 2: Pareto Frontier — Worst-Case Fairness Gap vs AUC\n"
#                  "(sweep pareto_w ∈ [0.00–0.30]; lower-left = ideal)",
#                  fontsize=13, fontweight="bold", y=1.01)
#     eo_patch = mpatches.Patch(color="#2dc653", label="AGAD-EO")
#     dp_patch = mpatches.Patch(color="#1a7c34", label="AGAD-DP")
#     for col, ds_key in enumerate(RUN_DATASETS):
#         for row, metric in enumerate(["eo", "dp"]):
#             ax   = axes[row][col]
#             pts  = pareto_curves[ds_key][metric]
#             aucs = [p["auc"]         for p in pts]
#             wcs  = [p["wc_fairness"] for p in pts]
#             pws  = [p["pareto_w"]    for p in pts]
#             color = "#2dc653" if metric == "eo" else "#1a7c34"
#             ax.plot(aucs, wcs, "-o", color=color, linewidth=2, markersize=5, zorder=3)
#             for auc, wc, pw in zip(aucs, wcs, pws):
#                 ax.annotate(f"{pw:.2f}", (auc, wc), textcoords="offset points",
#                             xytext=(4, 3), fontsize=6, color="#555555")
#             for bname, bcol in [
#                 ("adv_eo" if metric=="eo" else "adv_dp", "#4e9af1"),
#                 ("sgpen_eo" if metric=="eo" else "sgpen_dp", "#f4a261")
#             ]:
#                 r = all_results[ds_key][bname]
#                 ax.scatter(r["auc"], r[f"wc_{metric}"],
#                            marker="D", s=50, color=bcol, zorder=4, label=METHOD_LABELS[bname])
#             ax.scatter(all_results[ds_key]["vanilla"]["auc"],
#                        all_results[ds_key]["vanilla"][f"wc_{metric}"],
#                        marker="x", s=60, color="#6c757d", zorder=4,
#                        linewidths=1.5, label="Vanilla NN")
#             ax.set_title(f"{DS_LABELS[ds_key]}\nWC-{'EO' if metric=='eo' else 'DP'} vs AUC",
#                          fontsize=9)
#             ax.set_xlabel("AUC" if row == 1 else "", fontsize=8)
#             ax.set_ylabel("WC-Fairness Gap" if col == 0 else "", fontsize=8)
#             ax.yaxis.grid(True, linestyle="--", alpha=0.3); ax.set_axisbelow(True)
#             if col == 3 and row == 0:
#                 ax.legend(fontsize=6, loc="upper right")
#     fig.legend(handles=[eo_patch, dp_patch], loc="lower center", ncol=2,
#                fontsize=9, frameon=False, title="AGAD sweep", title_fontsize=9)
#     plt.tight_layout()
#     path = f"{PLOT_DIR}/fig2_pareto_curves.png"
#     plt.savefig(path, bbox_inches="tight", dpi=150); plt.close()
#     print(f"  Saved → {path}")

# def fig3_violin_stability():
#     fig, axes = plt.subplots(2, 4, figsize=(16, 7), sharey=False)
#     fig.suptitle("Figure 3: Seed Stability — WC-Fairness Gaps\n"
#                  "(narrower = more stable)", fontsize=13, fontweight="bold", y=1.01)
#     for col, ds_key in enumerate(RUN_DATASETS):
#         for row, metric in enumerate(["eo", "dp"]):
#             ax = axes[row][col]; wc_key = f"wc_{metric}_all"
#             data   = [all_results[ds_key][m].get(wc_key, []) for m in METHOD_ORDER]
#             data   = [d if d else [0.0] for d in data]
#             colors = [PALETTE[m] for m in METHOD_ORDER]
#             vp = ax.violinplot(data, positions=np.arange(len(METHOD_ORDER)),
#                                showmeans=True, showmedians=False, widths=0.6)
#             for body, c in zip(vp["bodies"], colors):
#                 body.set_facecolor(c); body.set_alpha(0.6)
#             vp["cmeans"].set_color("black"); vp["cmeans"].set_linewidth(1.5)
#             ax.set_xticks(np.arange(len(METHOD_ORDER)))
#             ax.set_xticklabels([METHOD_LABELS[m] for m in METHOD_ORDER],
#                                rotation=45, ha="right", fontsize=7)
#             ax.set_title(f"{DS_LABELS[ds_key]}\nWC-{'EO' if metric=='eo' else 'DP'}", fontsize=9)
#             ax.set_ylabel("Gap" if col == 0 else "", fontsize=9)
#             ax.yaxis.grid(True, linestyle="--", alpha=0.4); ax.set_axisbelow(True)
#     plt.tight_layout()
#     path = f"{PLOT_DIR}/fig3_seed_stability_violins.png"
#     plt.savefig(path, bbox_inches="tight", dpi=150); plt.close()
#     print(f"  Saved → {path}")

# def fig4_fg_bars():
#     fig, axes = plt.subplots(2, 4, figsize=(16, 7), sharey=False)
#     fig.suptitle("Figure 4: Fine-Grained (Top-K Subgroup) Fairness Gaps\n"
#                  "(mean gap over K=5 worst subgroups; lower is better)",
#                  fontsize=13, fontweight="bold", y=1.01)
#     for col, ds_key in enumerate(RUN_DATASETS):
#         for row, metric in enumerate(["eo", "dp"]):
#             ax = axes[row][col]; fg_key = f"fg_{metric}"
#             means  = [all_results[ds_key][m][fg_key]          for m in METHOD_ORDER]
#             stds   = [all_results[ds_key][m][fg_key + "_std"] for m in METHOD_ORDER]
#             colors = [PALETTE[m] for m in METHOD_ORDER]
#             x = np.arange(len(METHOD_ORDER))
#             bars = ax.bar(x, means, yerr=stds, color=colors, alpha=0.85,
#                           capsize=3, error_kw={"linewidth": 1.2})
#             ax.set_xticks(x)
#             ax.set_xticklabels([METHOD_LABELS[m] for m in METHOD_ORDER],
#                                rotation=45, ha="right", fontsize=7)
#             ax.set_title(f"{DS_LABELS[ds_key]}\nFG-{'EO' if metric=='eo' else 'DP'}", fontsize=9)
#             ax.set_ylabel("Gap" if col == 0 else "", fontsize=9)
#             ax.yaxis.grid(True, linestyle="--", alpha=0.4); ax.set_axisbelow(True)
#             for i, m in enumerate(METHOD_ORDER):
#                 if "agad" in m:
#                     bars[i].set_edgecolor("black"); bars[i].set_linewidth(1.5)
#     plt.tight_layout()
#     path = f"{PLOT_DIR}/fig4_fg_fairness_bars.png"
#     plt.savefig(path, bbox_inches="tight", dpi=150); plt.close()
#     print(f"  Saved → {path}")

# fig1_bar_fairness()
# fig2_pareto_curves()
# fig3_violin_stability()
# fig4_fg_bars()


# # ════════════════════════════════════════════════════════════════════════════
# #  Global summary table
# # ════════════════════════════════════════════════════════════════════════════

# print_section("GLOBAL SUMMARY TABLE — all datasets × methods")

# METRICS_SHOW = [
#     ("wc_eo","WC-EO"), ("wc_dp","WC-DP"),
#     ("fg_eo","FG-EO"), ("fg_dp","FG-DP"),
#     ("acc","Acc"),     ("auc","AUC"),
# ]

# header = f"  {'Dataset':<22}  {'Method':<20}"
# for _, label in METRICS_SHOW:
#     header += f"  {label:>16}"
# print(header)
# print("  " + "─" * (22 + 20 + 16 * len(METRICS_SHOW) + 4))

# for ds_key in RUN_DATASETS:
#     for i, m in enumerate(METHOD_ORDER):
#         r        = all_results[ds_key][m]
#         ds_label = DS_LABELS[ds_key] if i == 0 else ""
#         row      = f"  {ds_label:<22}  {METHOD_LABELS[m]:<20}"
#         for k, _ in METRICS_SHOW:
#             cell = f"{r[k]:.4f}±{r[k+'_std']:.4f}"
#             row += f"  {cell:>16}"
#         print(row)
#     print()


# # ════════════════════════════════════════════════════════════════════════════
# #  Save results
# # ════════════════════════════════════════════════════════════════════════════

# def clean(obj):
#     if isinstance(obj, dict):          return {k: clean(v) for k, v in obj.items()}
#     if isinstance(obj, (list, tuple)): return [clean(v) for v in obj]
#     if isinstance(obj, float) and (np.isnan(obj) or np.isinf(obj)): return None
#     if isinstance(obj, (np.floating, np.integer)): return obj.item()
#     if isinstance(obj, np.ndarray):   return obj.tolist()
#     return obj

# total_time = time.time() - t0
# payload = {
#     "results"       : all_results,
#     "best_params"   : BEST_PARAMS,
#     "pareto_curves" : pareto_curves,
#     "runtime"       : {"total_min": total_time / 60, "phase4_min": phase4_time / 60},
#     "fixes_applied" : {
#         "fix1_epoch_eo_accumulator" : "Epoch-level cumulative TPR tracking for sparse EO subgroups",
#         "fix2_wmw_auc_proxy"        : "Wilcoxon-Mann-Whitney differentiable AUC proxy",
#         "fix3_unified_adv_forward"  : "Removed forward_no_grl; unified forward(z, alpha)",
#     },
# }
# out_path = f"{WORK_DIR}/agad_hc_fix3_results.json"
# with open(out_path, "w") as f:
#     json.dump(clean(payload), f, indent=2)

# print_section("RUNTIME SUMMARY")
# print(f"  Total                : {total_time/60:.1f} min")
# print(f"  Phase 4 (Final run)  : {phase4_time/60:.1f} min")
# print(f"\n  Results  → {out_path}")
# print(f"  Pareto   → {WORK_DIR}/agad_pareto_curves_hc_fix3.json")
# print(f"  Plots    → {PLOT_DIR}/")
# print(f"             fig1_wc_fairness_bars.png")
# print(f"             fig2_pareto_curves.png")
# print(f"             fig3_seed_stability_violins.png")
# print(f"             fig4_fg_fairness_bars.png")

# print()
# print("=" * 70)
# print("  FIX VALIDATION CHECKLIST")
# print("=" * 70)
# print("  FIX 1 — ACS Employment EO (epoch-level TPR accumulator):")
# emp_eo   = all_results.get("acs_employment", {}).get("agad_eo_tuned", {})
# sgpen_eo = all_results.get("acs_employment", {}).get("sgpen_eo", {})
# if emp_eo and sgpen_eo:
#     delta  = sgpen_eo.get("wc_eo", 0) - emp_eo.get("wc_eo", 0)
#     status = "✓ FIXED" if delta > 0 else "✗ STILL FAILING"
#     print(f"    AGAD-EO WC-EO={emp_eo.get('wc_eo',0):.4f}  "
#           f"SGPen-EO WC-EO={sgpen_eo.get('wc_eo',0):.4f}  "
#           f"Δ={delta:+.5f}  {status}")

# print("  FIX 2 — Pareto sweep (WMW proxy, ACS Employment EO as example):")
# emp_curve = pareto_curves.get("acs_employment", {}).get("eo", [])
# if emp_curve:
#     wcs    = [p["wc_fairness"] for p in emp_curve]
#     aucs   = [p["auc"]         for p in emp_curve]
#     spread = max(wcs) - min(wcs)
#     auc_spread = max(aucs) - min(aucs)
#     status = "✓ CURVES ARE REAL" if spread > 5e-4 else "✗ STILL FLAT"
#     print(f"    WC-EO spread: {spread:.5f}  AUC spread: {auc_spread:.5f}  {status}")
#     for p in emp_curve:
#         print(f"      pareto_w={p['pareto_w']:.2f}  "
#               f"WC-EO={p['wc_fairness']:.4f}  AUC={p['auc']:.4f}")

# print("  FIX 3 — Unified adv.forward(alpha): forward_no_grl removed ✓")

In [6]:
"""
AGAD v4 — FINAL CONSOLIDATED SCRIPT
"""

# ════════════════════════════════════════════════════════════════════════════
#  Imports & setup
# ════════════════════════════════════════════════════════════════════════════

import subprocess
subprocess.run(["pip", "install", "folktables", "ucimlrepo", "--quiet",
                "--break-system-packages"], capture_output=True)

import os, gc, json, time, copy, random, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

warnings.filterwarnings("ignore")

WORK_DIR = "/kaggle/working"
PLOT_DIR = os.path.join(WORK_DIR, "agad_plots_final")
os.makedirs(PLOT_DIR, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SEEDS_FINAL    = [42, 123, 7]
RUN_DATASETS   = ["adult", "acs_income", "acs_employment", "communities_crime"]
PARETO_SWEEP_W = [0.00, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30]

# ── Architecture / training constants ──────────────────────────────────────
HIDDEN_DIM   = 128
DROPOUT      = 0.25
LR           = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE     = 35
GRL_MAX      = 1.0
ADV_STEPS    = 3
ADV_LR_MULT  = 2.0
LAMBDA_MAX   = 2.0
PHASE1_END   = 10
PHASE2_END   = 20
MIN_SG_N     = 40
MIN_SG_FRAC  = 0.03
MAX_SG_FRAC  = 0.50
MAX_DEPTH    = 3
TOP_K_REPORT = 5
MIN_POS_FOR_EO       = 2
AUC_PROXY_SHARPNESS  = 10.0
AUC_PROXY_SUBSAMPLE  = 64

FULL_CFG = dict(
    adult             = dict(epochs=80,  batch=2048),
    acs_income        = dict(epochs=120, batch=4096),
    acs_employment    = dict(epochs=140, batch=4096),
    communities_crime = dict(epochs=130, batch=256),
)

VANILLA_ACC = dict(
    adult             = 0.7886,
    acs_income        = 0.7978,
    acs_employment    = 0.8075,
    communities_crime = 0.7118,
)
ACC_TOLERANCE = 0.05

SGPEN_GAMMA = dict(
    adult             = 0.20,
    acs_income        = 0.30,
    acs_employment    = 0.50,
    communities_crime = 0.10,
)

# ════════════════════════════════════════════════════════════════════════════
#  BEST HYPERPARAMETERS (all hardcoded — no Optuna needed)
#
#  Sources:
#    adult            eo/dp : fix3 grid search
#    acs_income       eo    : FIX 8a  (top_k_loss 2→6, alpha 12.82→15.0,
#                                      lambda0 0.59→0.55, pareto_w 0.14→0.0)
#    acs_income       dp    : fix4 grid search
#    acs_employment   eo    : FIX 8b  (alpha 20→12, lambda0 0.95→0.75,
#                                      top_k_loss 1→3, score_mode→with_auc,
#                                      min_ckpt_epoch=15)
#    acs_employment   dp    : fix5 (unchanged)
#    communities_crime eo   : fix3 grid search
#    communities_crime dp   : fix5 Optuna (n=25)
# ════════════════════════════════════════════════════════════════════════════

BEST_PARAMS = {
    "adult": {
        "eo": dict(gamma_sg=0.42, lambda0=0.598, alpha=14.48, pareto_w=0.07,
                   task_weight=1.85, top_k_loss=1, sg_warmup=10),
        "dp": dict(gamma_sg=0.36, lambda0=0.65,  alpha=14.48, pareto_w=0.13,
                   task_weight=1.85, top_k_loss=1, sg_warmup=10),
    },
    "acs_income": {
        # FIX 8a: top_k_loss 2→6, alpha 12.82→15.0, lambda0→0.55, pareto_w→0.0
        "eo": dict(gamma_sg=0.18, lambda0=0.55, alpha=15.0,  pareto_w=0.0,
                   task_weight=1.4,  top_k_loss=6, sg_warmup=10),
        # fix4 params (unchanged)
        "dp": dict(gamma_sg=0.18, lambda0=0.74, alpha=13.0,  pareto_w=0.08,
                   task_weight=1.68, top_k_loss=2, sg_warmup=10),
    },
    "acs_employment": {
        # FIX 8b: alpha 20→12, lambda0 0.95→0.75, top_k_loss 1→3,
        #         score_mode=with_auc, min_ckpt_epoch=15
        "eo": dict(gamma_sg=0.0,  lambda0=0.75, alpha=12.0,  pareto_w=0.0,
                   task_weight=1.8,  top_k_loss=3, sg_warmup=3),
        # fix5 (unchanged)
        "dp": dict(gamma_sg=0.18, lambda0=0.40, alpha=10.0,  pareto_w=0.053,
                   task_weight=1.55, top_k_loss=1, sg_warmup=15),
    },
    "communities_crime": {
        # fix3 (unchanged)
        "eo": dict(gamma_sg=0.35, lambda0=0.49, alpha=12.36, pareto_w=0.04,
                   task_weight=1.92, top_k_loss=4, sg_warmup=24),
        # fix5 Optuna result (hardcoded)
        "dp": dict(gamma_sg=0.45, lambda0=0.80, alpha=17.0,  pareto_w=0.0,
                   task_weight=1.8,  top_k_loss=4, sg_warmup=12),
    },
}

# Per (dataset, metric): extra weight on FG-k in checkpoint score
FG_CKPT_WEIGHT = {
    "adult":             {"eo": 0.0,  "dp": 0.0},
    "acs_income":        {"eo": 0.3,  "dp": 0.0},   # fix8a: helps pick ckpt with low FG
    "acs_employment":    {"eo": 0.0,  "dp": 0.0},
    "communities_crime": {"eo": 0.0,  "dp": 0.0},
}

# Per (dataset, metric): minimum epoch before checkpointing begins
# FIX 8b: employment EO was always selecting epoch-0 (untrained) weights
# because V_t is tiny before fairness pressure kicks in.
MIN_CKPT_EPOCH = {
    "adult":             {"eo": 0,  "dp": 0},
    "acs_income":        {"eo": 10, "dp": 10},
    "acs_employment":    {"eo": 15, "dp": 15},
    "communities_crime": {"eo": 0,  "dp": 0},
}

# score_mode: "with_auc" everywhere — "fairness_only" caused epoch-0 bug (fix8b)
SCORE_MODE = {
    "adult":             {"eo": "with_auc", "dp": "with_auc"},
    "acs_income":        {"eo": "with_auc", "dp": "with_auc"},
    "acs_employment":    {"eo": "with_auc", "dp": "with_auc"},
    "communities_crime": {"eo": "with_auc", "dp": "with_auc"},
}

DEFAULT_HPARAMS = dict(gamma_sg=0.20, lambda0=0.30, alpha=8.0, pareto_w=0.12,
                       task_weight=2.0, top_k_loss=3, sg_warmup=20)

PALETTE = {
    "vanilla"       : "#6c757d",
    "adv_eo"        : "#4e9af1",
    "adv_dp"        : "#1a6fc4",
    "sgpen_eo"      : "#f4a261",
    "sgpen_dp"      : "#e76f51",
    "agad_eo_tuned" : "#2dc653",
    "agad_dp_tuned" : "#1a7c34",
}

METHOD_LABELS = {
    "vanilla"       : "Vanilla NN",
    "adv_eo"        : "Adv-EO",
    "adv_dp"        : "Adv-DP",
    "sgpen_eo"      : "SGPen-EO",
    "sgpen_dp"      : "SGPen-DP",
    "agad_eo_tuned" : "AGAD-EO ★",
    "agad_dp_tuned" : "AGAD-DP ★",
}
METHOD_ORDER = list(METHOD_LABELS.keys())

DS_LABELS = {
    "adult"            : "Adult Income",
    "acs_income"       : "ACS Income",
    "acs_employment"   : "ACS Employment",
    "communities_crime": "Communities & Crime",
}

print("=" * 70)
print("  AGAD v4 — FINAL CONSOLIDATED SCRIPT")
print("=" * 70)
print(f"  Device : {DEVICE}")
print(f"  Seeds  : {SEEDS_FINAL}")
print()
print("  FIXES APPLIED:")
print("    FIX 1 — Epoch-level EO accumulator")
print("    FIX 2 — WMW differentiable AUC proxy")
print("    FIX 3 — Unified adv.forward(alpha=0)")
print("    FIX 4 — ACS Income DP: grid-search params")
print("    FIX 5 — Communities & Crime DP: Optuna params (hardcoded)")
print("    FIX 8a — ACS Income EO: top_k_loss=6, re-tuned alpha/lambda0")
print("    FIX 8b — ACS Employment EO: min_ckpt_epoch=15 (epoch-0 bug)")
print("             + alpha 20→12, lambda0 0.95→0.75, score_mode=with_auc")
print("=" * 70)


# ════════════════════════════════════════════════════════════════════════════
#  Utilities
# ════════════════════════════════════════════════════════════════════════════

def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def ensure_binary(sv):
    sv = np.asarray(sv).ravel(); u = np.unique(sv)
    if len(u) <= 1: return np.zeros(len(sv), int)
    if len(u) == 2: return (sv == u[1]).astype(int)
    maj = u[np.argmax([(sv == v).sum() for v in u])]
    return (sv != maj).astype(int)

def safe_auc(yt, yp):
    try:    return float(roc_auc_score(yt, yp)) if len(np.unique(yt)) >= 2 else float("nan")
    except: return float("nan")

def strat_split(X, y, sens_list, test_size=0.2, seed=42):
    key = np.array(y).astype(str)
    for s in sens_list:
        key = key + "_" + np.array(s).astype(str)
    u, c = np.unique(key, return_counts=True)
    key[np.isin(key, u[c < 2])] = np.array(y)[np.isin(key, u[c < 2])].astype(str)
    try:
        return train_test_split(np.arange(len(y)), test_size=test_size,
                                stratify=key, random_state=seed)
    except:
        return train_test_split(np.arange(len(y)), test_size=test_size,
                                stratify=y, random_state=seed)

def eo_gap(y_true, y_prob, mask):
    sg_t = y_true[mask]; sg_p = y_prob[mask]
    ot_t = y_true[~mask]; ot_p = y_prob[~mask]
    if len(np.unique(sg_t)) < 2 or len(np.unique(ot_t)) < 2: return 0.0
    tpr_sg = sg_p[sg_t == 1].mean() if (sg_t == 1).sum() > 0 else 0.0
    tpr_ot = ot_p[ot_t == 1].mean() if (ot_t == 1).sum() > 0 else 0.0
    fpr_sg = sg_p[sg_t == 0].mean() if (sg_t == 0).sum() > 0 else 0.0
    fpr_ot = ot_p[ot_t == 0].mean() if (ot_t == 0).sum() > 0 else 0.0
    return float(max(abs(tpr_sg - tpr_ot), abs(fpr_sg - fpr_ot)))

def dp_gap(y_prob, mask):
    return float(abs(y_prob[mask].mean() - y_prob.mean()))

def fg_metric(y_true, y_prob, sgs, k=TOP_K_REPORT, metric="eo"):
    gaps = []
    for sg in sgs:
        mem = sg["mem"]
        if len(np.unique(y_true[mem])) < 2: continue
        g = eo_gap(y_true, y_prob, mem) if metric == "eo" else dp_gap(y_prob, mem)
        gaps.append(g)
    if not gaps: return 0.0
    gaps.sort(reverse=True)
    return float(np.mean(gaps[:k]))

def get_depth_limit(epoch):
    if epoch < PHASE1_END: return 1
    if epoch < PHASE2_END: return 2
    return 3

def enumerate_subgroups(sv_bin_list, attr_names, n, max_depth=2):
    n_attr = len(attr_names)
    min_n  = max(MIN_SG_N, int(MIN_SG_FRAC * n))
    max_n  = int(MAX_SG_FRAC * n)
    sgs, seen = [], set()
    for mask in range(1, 2 ** n_attr):
        active = [i for i in range(n_attr) if mask & (1 << i)]
        if len(active) > max_depth: continue
        for vals in range(2 ** len(active)):
            req = [(vals >> j) & 1 for j in range(len(active))]
            mem = np.ones(n, dtype=bool)
            for ai, rv in zip(active, req):
                mem &= (sv_bin_list[ai] == rv)
            key = mem.tobytes()
            if key in seen: continue
            seen.add(key)
            sz = int(mem.sum())
            if sz < min_n or sz > max_n: continue
            spec = [(attr_names[i], req[j]) for j, i in enumerate(active)]
            sgs.append({"mem": mem, "spec": spec})
    return sgs

def audit(epoch, y_val, p_val, sv_val_bin, attr_names, metric="eo"):
    depth  = min(get_depth_limit(epoch), MAX_DEPTH)
    n      = len(p_val)
    sgs    = enumerate_subgroups(sv_val_bin, attr_names, n, max_depth=depth)
    ranked = []
    for sg in sgs:
        mem = sg["mem"]
        if len(np.unique(y_val[mem])) < 2: continue
        gap = eo_gap(y_val, p_val, mem) if metric == "eo" else dp_gap(p_val, mem)
        ranked.append((gap, sg["spec"], mem))
    ranked.sort(key=lambda x: -x[0])
    worst_gap  = ranked[0][0] if ranked else 0.0
    worst_spec = ranked[0][1] if ranked else None
    topk_specs = [r[1] for r in ranked[:6]]
    fg_k       = float(np.mean([r[0] for r in ranked[:TOP_K_REPORT]])) if ranked else 0.0
    return float(worst_gap), worst_spec, float(fg_k), topk_specs

def adaptive_lambda(V_t, lambda0, alpha):
    return min(lambda0 * (1.0 + alpha * V_t), LAMBDA_MAX)

def build_intersection_labels(sv_bin_list, n_attrs):
    n      = len(sv_bin_list[0])
    labels = np.zeros(n, dtype=np.int64)
    for sv in sv_bin_list:
        labels = labels * 2 + np.asarray(sv, dtype=np.int64)
    return labels

def spec_to_train_mask(spec, attr_names, sv_tr_np, n_tr):
    wm = np.ones(n_tr, dtype=bool)
    for (attr_name, val) in spec:
        if attr_name in attr_names:
            ai = attr_names.index(attr_name)
            wm &= (sv_tr_np[ai] == val)
    return wm

def evaluate(proba, y, sv_bin_list, attr_names, tag="", verbose=True):
    pred  = (proba > 0.5).astype(int)
    acc   = float(accuracy_score(y, pred))
    auc   = safe_auc(y, proba)
    n     = len(proba)
    sgs   = enumerate_subgroups(sv_bin_list, attr_names, n, max_depth=MAX_DEPTH)
    wc_eo = max((eo_gap(y, proba, sg["mem"])
                 for sg in sgs if len(np.unique(y[sg["mem"]])) >= 2), default=0.0)
    wc_dp = max((dp_gap(proba, sg["mem"]) for sg in sgs), default=0.0)
    fg_eo = fg_metric(y, proba, sgs, k=TOP_K_REPORT, metric="eo")
    fg_dp = fg_metric(y, proba, sgs, k=TOP_K_REPORT, metric="dp")
    if verbose:
        print(f"    [{tag:<32}]  WC-EO={wc_eo:.4f}  WC-DP={wc_dp:.4f}  "
              f"FG-EO={fg_eo:.4f}  FG-DP={fg_dp:.4f}  Acc={acc:.4f}  AUC={auc:.4f}")
    return {"wc_eo": wc_eo, "wc_dp": wc_dp, "fg_eo": fg_eo,
            "fg_dp": fg_dp, "acc": acc, "auc": auc}

def aggregate_seeds(results_list):
    keys = ["wc_eo", "wc_dp", "fg_eo", "fg_dp", "acc", "auc"]
    agg  = {}
    for k in keys:
        vals = [r[k] for r in results_list if k in r and r[k] is not None]
        agg[k]          = float(np.mean(vals)) if vals else float("nan")
        agg[k + "_std"] = float(np.std(vals))  if len(vals) > 1 else 0.0
        agg[k + "_all"] = [float(v) for v in vals]
    return agg

def print_section(title, width=70):
    print(); print("=" * width); print(f"  {title}"); print("=" * width)

def print_subsection(title, width=70):
    print(); print("-" * width); print(f"  {title}"); print("-" * width)


# ════════════════════════════════════════════════════════════════════════════
#  FIX 2: WMW differentiable AUC proxy
# ════════════════════════════════════════════════════════════════════════════

def wmw_auc_proxy(prob, y_binary, sharpness=AUC_PROXY_SHARPNESS,
                  n_sub=AUC_PROXY_SUBSAMPLE):
    pos_idx = (y_binary == 1).nonzero(as_tuple=True)[0]
    neg_idx = (y_binary == 0).nonzero(as_tuple=True)[0]
    if len(pos_idx) == 0 or len(neg_idx) == 0:
        return torch.tensor(0.5, device=prob.device)
    if len(pos_idx) > n_sub:
        perm = torch.randperm(len(pos_idx), device=prob.device)[:n_sub]
        pos_idx = pos_idx[perm]
    if len(neg_idx) > n_sub:
        perm = torch.randperm(len(neg_idx), device=prob.device)[:n_sub]
        neg_idx = neg_idx[perm]
    p_pos = prob[pos_idx]
    p_neg = prob[neg_idx]
    diff  = p_pos.unsqueeze(1) - p_neg.unsqueeze(0)
    return torch.sigmoid(sharpness * diff).mean()


# ════════════════════════════════════════════════════════════════════════════
#  Model components
# ════════════════════════════════════════════════════════════════════════════

class GRL(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha; return x.clone()
    @staticmethod
    def backward(ctx, g):
        return -ctx.alpha * g, None

class GradReversal(nn.Module):
    def __init__(self, alpha=1.0):
        super().__init__(); self.alpha = alpha
    def forward(self, x):
        return GRL.apply(x, self.alpha)

class Encoder(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        h = HIDDEN_DIM
        self.rep_dim = h // 2 + 32
        self.net = nn.Sequential(
            nn.Linear(in_dim, h),         nn.BatchNorm1d(h),            nn.GELU(), nn.Dropout(DROPOUT),
            nn.Linear(h, 256),            nn.BatchNorm1d(256),          nn.GELU(), nn.Dropout(DROPOUT),
            nn.Linear(256, 128),          nn.BatchNorm1d(128),          nn.GELU(), nn.Dropout(DROPOUT * 0.5),
            nn.Linear(128, self.rep_dim), nn.BatchNorm1d(self.rep_dim), nn.GELU(), nn.Dropout(DROPOUT * 0.5),
        )
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight); nn.init.zeros_(m.bias)
    def forward(self, x): return self.net(x)

class PredictorHead(nn.Module):
    def __init__(self, rep_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(rep_dim, rep_dim // 2), nn.GELU(),
            nn.Linear(rep_dim // 2, 1),
        )
    def forward(self, z): return self.net(z).squeeze(1)

class IntersectionAdversaryHead(nn.Module):
    def __init__(self, rep_dim, n_marginal_attrs, n_intersection_classes):
        super().__init__()
        h = HIDDEN_DIM // 2
        self.grl   = GradReversal(1.0)
        self.trunk = nn.Sequential(
            nn.Linear(rep_dim, h), nn.GELU(), nn.Dropout(DROPOUT),
            nn.Linear(h, h // 2), nn.GELU(),
        )
        self.marginal_out     = nn.Linear(h // 2, n_marginal_attrs)
        self.intersection_out = nn.Linear(h // 2, n_intersection_classes)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight); nn.init.zeros_(m.bias)

    def set_alpha(self, a):
        self.grl.alpha = a

    # FIX 3: alpha=None uses stored value, alpha=0 detaches adversary cleanly
    def forward(self, z, alpha=None):
        if alpha is not None:
            self.grl.alpha = alpha
        h = self.trunk(self.grl(z))
        return self.marginal_out(h), self.intersection_out(h)


# ════════════════════════════════════════════════════════════════════════════
#  FIX 1: Epoch-level EO accumulator
#  Accumulates TPR statistics across all batches in an epoch so the
#  subgroup penalty has a stable, low-variance gradient signal.
# ════════════════════════════════════════════════════════════════════════════

class EpochEOAccumulator:
    def __init__(self, n_masks):
        self.n_masks = n_masks
        self.reset()

    def reset(self):
        self.sg_pos_prob_sum  = [0.0] * self.n_masks
        self.sg_pos_count     = [0]   * self.n_masks
        self.all_pos_prob_sum = 0.0
        self.all_pos_count    = 0

    def update(self, prob_np, y_np, batch_sg_masks_np):
        pos_mask = (y_np == 1)
        self.all_pos_prob_sum += float(prob_np[pos_mask].sum()) if pos_mask.any() else 0.0
        self.all_pos_count    += int(pos_mask.sum())
        for i, sg_mask in enumerate(batch_sg_masks_np):
            sg_pos = sg_mask & pos_mask
            self.sg_pos_prob_sum[i] += float(prob_np[sg_pos].sum()) if sg_pos.any() else 0.0
            self.sg_pos_count[i]    += int(sg_pos.sum())

    def compute_penalties(self):
        if self.all_pos_count == 0:
            return {i: 0.0 for i in range(self.n_masks)}
        tpr_all = self.all_pos_prob_sum / max(self.all_pos_count, 1)
        penalties = {}
        for i in range(self.n_masks):
            if self.sg_pos_count[i] == 0:
                penalties[i] = 0.0
            else:
                tpr_sg = self.sg_pos_prob_sum[i] / self.sg_pos_count[i]
                penalties[i] = abs(tpr_sg - tpr_all)
        return penalties


# ════════════════════════════════════════════════════════════════════════════
#  AGAD training core
#
#  Parameters:
#    score_mode      : "with_auc" (default)
#    fg_ckpt_weight  : blend FG into checkpoint score (helps acs_income EO)
#    min_ckpt_epoch  : FIX 8b — don't checkpoint until this epoch
#                      (prevents epoch-0 untrained weights from always winning)
# ════════════════════════════════════════════════════════════════════════════

def run_agad_with_hparams(d, cfg, metric, hp, seed=42,
                           verbose=False, epoch_verbose=False,
                           score_mode="with_auc",
                           fg_ckpt_weight=0.0,
                           min_ckpt_epoch=0):
    set_seed(seed)
    gamma_sg    = hp["gamma_sg"]
    lambda0     = hp["lambda0"]
    alpha_val   = hp["alpha"]
    pareto_w    = hp["pareto_w"]
    task_weight = hp["task_weight"]
    top_k_loss  = hp["top_k_loss"]
    sg_warmup   = hp["sg_warmup"]

    n_attrs = len(d["attr_names"])
    n_inter = 2 ** n_attrs

    Xt   = torch.tensor(d["X_tr"],  dtype=torch.float32, device=DEVICE)
    yt   = torch.tensor(d["y_tr"],  dtype=torch.float32, device=DEVICE)
    Xv   = torch.tensor(d["X_val"], dtype=torch.float32, device=DEVICE)
    Xte  = torch.tensor(d["X_te"],  dtype=torch.float32, device=DEVICE)
    sv_t = [torch.tensor(sv, dtype=torch.float32, device=DEVICE) for sv in d["sv_tr"]]
    inter_labels_np = build_intersection_labels(d["sv_tr"], n_attrs)
    inter_t  = torch.tensor(inter_labels_np, dtype=torch.long, device=DEVICE)
    idx_t    = torch.arange(len(d["y_tr"]), dtype=torch.long)
    sv_tr_np = [np.asarray(sv) for sv in d["sv_tr"]]
    n_tr     = len(d["y_tr"])

    enc  = Encoder(d["X_tr"].shape[1]).to(DEVICE)
    head = PredictorHead(enc.rep_dim).to(DEVICE)
    adv  = IntersectionAdversaryHead(enc.rep_dim, n_attrs, n_inter).to(DEVICE)
    bce  = nn.BCEWithLogitsLoss()
    ce   = nn.CrossEntropyLoss()
    eps  = torch.tensor(1e-4, device=DEVICE)

    opt_enc = optim.AdamW(
        list(enc.parameters()) + list(head.parameters()),
        lr=LR, weight_decay=WEIGHT_DECAY)
    opt_adv = optim.AdamW(adv.parameters(),
                          lr=LR * ADV_LR_MULT, weight_decay=WEIGHT_DECAY)
    sched  = optim.lr_scheduler.CosineAnnealingLR(
        opt_enc, cfg["epochs"], eta_min=LR * 0.01)
    loader = DataLoader(
        TensorDataset(Xt, yt, idx_t),
        batch_size=cfg["batch"], shuffle=True, drop_last=True)

    best_score    = np.inf
    best_enc      = best_head = None
    no_imp        = 0
    V_t           = 0.0
    topk_masks_tr    = []
    topk_masks_tr_np = []
    acc_floor     = VANILLA_ACC[d["ds_key"]] - ACC_TOLERANCE

    eo_accum        = None
    epoch_penalties = {}

    for epoch in range(cfg["epochs"]):
        enc.eval(); head.eval()
        with torch.no_grad():
            pv_scan = torch.sigmoid(head(enc(Xv))).cpu().numpy()

        V_t, worst_spec, fg_k, topk_specs = audit(
            epoch, d["y_val"], pv_scan, d["sv_val"], d["attr_names"], metric=metric)
        lam_t = adaptive_lambda(V_t, lambda0, alpha_val)

        if epoch >= sg_warmup and topk_specs:
            topk_masks_tr = [
                torch.tensor(
                    spec_to_train_mask(spec, d["attr_names"], sv_tr_np, n_tr),
                    dtype=torch.bool, device=DEVICE)
                for spec in topk_specs[:top_k_loss]
            ]
            topk_masks_tr_np = [m.cpu().numpy() for m in topk_masks_tr]
            if eo_accum is None or eo_accum.n_masks != len(topk_masks_tr):
                eo_accum = EpochEOAccumulator(len(topk_masks_tr))
                epoch_penalties = {i: 0.0 for i in range(len(topk_masks_tr))}
        else:
            topk_masks_tr    = []
            topk_masks_tr_np = []

        grl_alpha = GRL_MAX * max(0.1, epoch / max(1, cfg["epochs"]))
        adv.set_alpha(grl_alpha)
        enc.train(); head.train(); adv.train()

        if eo_accum is not None:
            eo_accum.reset()

        for xb, yb, bidx in loader:
            # ── Adversary update (FIX 3: alpha=0 → no GRL in adv forward) ──
            z_d = enc(xb).detach()
            for _ in range(ADV_STEPS):
                opt_adv.zero_grad(set_to_none=True)
                m_out, i_out = adv(z_d, alpha=0)
                loss_a = (
                    sum(nn.functional.binary_cross_entropy_with_logits(
                        m_out[:, i], sv_t[i][bidx])
                        for i in range(n_attrs)) / n_attrs
                    + ce(i_out, inter_t[bidx]))
                loss_a.backward()
                nn.utils.clip_grad_norm_(adv.parameters(), 1.0)
                opt_adv.step()

            # ── Encoder + head update ──
            opt_enc.zero_grad(set_to_none=True)
            z      = enc(xb)
            logits = head(z)
            prob   = torch.sigmoid(logits)
            task_loss = bce(logits, yb)

            # FIX 2: WMW AUC proxy
            if pareto_w > 0:
                auc_proxy      = wmw_auc_proxy(prob, yb.long())
                auc_proxy_loss = 1.0 - auc_proxy
            else:
                auc_proxy_loss = torch.tensor(0.0, device=DEVICE)

            # ── Adversarial fairness loss ──
            fair_loss = torch.tensor(0.0, device=DEVICE)
            m_adv, i_adv = adv(z)
            for i, sv in enumerate(sv_t):
                tgt = sv[bidx]
                if metric == "eo":
                    sf = tgt.float(); yf = yb.float()
                    g0y1 = (1 - sf) * yf;       g1y1 = sf * yf
                    g0y0 = (1 - sf) * (1 - yf); g1y0 = sf * (1 - yf)
                    if all(g.sum() > 1e-6 for g in [g0y1, g1y1, g0y0, g1y0]):
                        tpr0 = (prob * g0y1).sum() / (g0y1.sum() + eps)
                        tpr1 = (prob * g1y1).sum() / (g1y1.sum() + eps)
                        fpr0 = (prob * g0y0).sum() / (g0y0.sum() + eps)
                        fpr1 = (prob * g1y0).sum() / (g1y0.sum() + eps)
                        fair_loss += torch.max(
                            torch.abs(tpr0 - tpr1), torch.abs(fpr0 - fpr1))
                else:
                    n0 = (1 - tgt).sum() + eps; n1 = tgt.sum() + eps
                    fair_loss += torch.abs(
                        (prob * (1 - tgt)).sum() / n0
                        - (prob * tgt).sum() / n1)
                fair_loss += nn.functional.binary_cross_entropy_with_logits(
                    m_adv[:, i], tgt)
            fair_loss += ce(i_adv, inter_t[bidx])

            # ── Subgroup penalty with epoch-level EO reweighting (FIX 1) ──
            sg_loss = torch.tensor(0.0, device=DEVICE)
            if topk_masks_tr:
                n_active = 0
                batch_probs_np = prob.detach().cpu().numpy()
                batch_y_np     = yb.cpu().numpy()
                if eo_accum is not None:
                    batch_sg_masks_np_batch = [
                        m[bidx.cpu().numpy()] for m in topk_masks_tr_np
                    ]
                    eo_accum.update(batch_probs_np, batch_y_np,
                                    batch_sg_masks_np_batch)

                for i, wm_tr in enumerate(topk_masks_tr):
                    batch_sg_mask = wm_tr[bidx]
                    if batch_sg_mask.sum() < 2: continue
                    ep_mult = 1.0 + epoch_penalties.get(i, 0.0)
                    if metric == "eo":
                        pos_sg  = batch_sg_mask & (yb == 1)
                        pos_all = (yb == 1)
                        if pos_sg.sum() >= MIN_POS_FOR_EO and \
                           pos_all.sum() > pos_sg.sum():
                            contrib = torch.abs(
                                prob[pos_sg].mean() - prob[pos_all].mean())
                        else:
                            contrib = torch.abs(
                                prob[batch_sg_mask].mean() - prob.mean())
                        sg_loss  += ep_mult * contrib
                        n_active += 1
                    else:
                        contrib  = torch.abs(
                            prob[batch_sg_mask].mean() - prob.mean())
                        sg_loss  += ep_mult * contrib
                        n_active += 1

                if n_active > 0:
                    sg_loss = sg_loss / n_active

            loss = (task_weight * task_loss
                    + lam_t * fair_loss / n_attrs
                    + gamma_sg * sg_loss
                    + pareto_w * auc_proxy_loss)

            loss.backward()
            nn.utils.clip_grad_norm_(
                list(enc.parameters()) + list(head.parameters()), 0.5)
            opt_enc.step()

        if eo_accum is not None and metric == "eo":
            epoch_penalties = eo_accum.compute_penalties()

        sched.step()

        enc.eval(); head.eval()
        with torch.no_grad():
            pv  = torch.sigmoid(head(enc(Xv))).cpu().numpy()
        acc = accuracy_score(d["y_val"], (pv > 0.5).astype(int))
        auc = safe_auc(d["y_val"], pv)

        # ── Checkpoint score ──
        # FIX 8b: min_ckpt_epoch prevents epoch-0 untrained weights from
        # always minimising score (V_t is near-zero before fairness kicks in).
        if score_mode == "fairness_only":
            score = V_t + fg_ckpt_weight * fg_k + 0.5 * max(0.0, acc_floor - acc)
        else:
            score = (V_t
                     + fg_ckpt_weight * fg_k
                     + pareto_w * (1 - auc if not np.isnan(auc) else 0)
                     + 0.5 * max(0.0, acc_floor - acc))

        if score < best_score and epoch >= min_ckpt_epoch:
            best_score = score
            best_enc   = copy.deepcopy(enc.state_dict())
            best_head  = copy.deepcopy(head.state_dict())
            no_imp     = 0
        elif epoch >= min_ckpt_epoch:
            no_imp += 1

        # Before min_ckpt_epoch: always save so best_enc is never None
        if epoch < min_ckpt_epoch:
            best_enc  = copy.deepcopy(enc.state_dict())
            best_head = copy.deepcopy(head.state_dict())

        if epoch_verbose and (epoch % 10 == 0 or epoch == cfg["epochs"] - 1):
            print(f"      epoch {epoch:03d}  V_t={V_t:.4f}  "
                  f"lam={lam_t:.3f}  acc={acc:.4f}  auc={auc:.4f}  "
                  f"score={score:.5f}  no_imp={no_imp}")

        if no_imp >= PATIENCE:
            if epoch_verbose:
                print(f"      [early stop at epoch {epoch}]")
            break

    enc.load_state_dict(best_enc)
    head.load_state_dict(best_head)
    enc.eval(); head.eval()
    with torch.no_grad():
        proba = torch.sigmoid(head(enc(Xte))).cpu().numpy()

    tag = f"AGAD-{metric.upper()} (final)"
    return evaluate(proba, d["y_te"], d["sv_te"], d["attr_names"],
                    tag=tag, verbose=verbose)


# ════════════════════════════════════════════════════════════════════════════
#  Baselines
# ════════════════════════════════════════════════════════════════════════════

def run_vanilla(d, cfg, seed=42):
    set_seed(seed)
    Xt  = torch.tensor(d["X_tr"],  dtype=torch.float32, device=DEVICE)
    yt  = torch.tensor(d["y_tr"],  dtype=torch.float32, device=DEVICE)
    Xv  = torch.tensor(d["X_val"], dtype=torch.float32, device=DEVICE)
    Xte = torch.tensor(d["X_te"],  dtype=torch.float32, device=DEVICE)
    enc  = Encoder(d["X_tr"].shape[1]).to(DEVICE)
    head = PredictorHead(enc.rep_dim).to(DEVICE)
    bce  = nn.BCEWithLogitsLoss()
    opt  = optim.AdamW(list(enc.parameters()) + list(head.parameters()),
                       lr=LR, weight_decay=WEIGHT_DECAY)
    sched  = optim.lr_scheduler.CosineAnnealingLR(opt, cfg["epochs"], eta_min=LR * 0.01)
    loader = DataLoader(TensorDataset(Xt, yt),
                        batch_size=cfg["batch"], shuffle=True, drop_last=True)
    best_score = np.inf; best_enc = best_head = None; no_imp = 0
    for epoch in range(cfg["epochs"]):
        enc.train(); head.train()
        for xb, yb in loader:
            opt.zero_grad(set_to_none=True)
            bce(head(enc(xb)), yb).backward()
            opt.step()
        sched.step()
        enc.eval(); head.eval()
        with torch.no_grad():
            pv = torch.sigmoid(head(enc(Xv))).cpu().numpy()
        auc = safe_auc(d["y_val"], pv)
        V, _, _, _ = audit(epoch, d["y_val"], pv, d["sv_val"], d["attr_names"])
        score = V + 0.12 * (1 - auc if not np.isnan(auc) else 0)
        if score < best_score:
            best_score = score; best_enc = copy.deepcopy(enc.state_dict())
            best_head  = copy.deepcopy(head.state_dict()); no_imp = 0
        else:
            no_imp += 1
        if no_imp >= PATIENCE: break
    enc.load_state_dict(best_enc); head.load_state_dict(best_head)
    enc.eval(); head.eval()
    with torch.no_grad():
        proba = torch.sigmoid(head(enc(Xte))).cpu().numpy()
    return evaluate(proba, d["y_te"], d["sv_te"], d["attr_names"], tag="Vanilla NN")


def _adv_static(d, cfg, metric, gamma_sg_static, seed=42):
    """Adversarial baseline (gamma_sg=0) or SubgroupPenalty (gamma_sg>0)."""
    set_seed(seed)
    n_attrs = len(d["attr_names"])
    n_inter = 2 ** n_attrs
    Xt   = torch.tensor(d["X_tr"],  dtype=torch.float32, device=DEVICE)
    yt   = torch.tensor(d["y_tr"],  dtype=torch.float32, device=DEVICE)
    Xv   = torch.tensor(d["X_val"], dtype=torch.float32, device=DEVICE)
    Xte  = torch.tensor(d["X_te"],  dtype=torch.float32, device=DEVICE)
    sv_t = [torch.tensor(sv, dtype=torch.float32, device=DEVICE) for sv in d["sv_tr"]]
    inter_labels_np = build_intersection_labels(d["sv_tr"], n_attrs)
    inter_t = torch.tensor(inter_labels_np, dtype=torch.long, device=DEVICE)
    idx_t   = torch.arange(len(d["y_tr"]), dtype=torch.long)
    enc  = Encoder(d["X_tr"].shape[1]).to(DEVICE)
    head = PredictorHead(enc.rep_dim).to(DEVICE)
    adv  = IntersectionAdversaryHead(enc.rep_dim, n_attrs, n_inter).to(DEVICE)
    bce  = nn.BCEWithLogitsLoss(); ce = nn.CrossEntropyLoss()
    eps  = torch.tensor(1e-4, device=DEVICE)
    opt_enc = optim.AdamW(list(enc.parameters()) + list(head.parameters()),
                          lr=LR, weight_decay=WEIGHT_DECAY)
    opt_adv = optim.AdamW(adv.parameters(), lr=LR * ADV_LR_MULT, weight_decay=WEIGHT_DECAY)
    sched  = optim.lr_scheduler.CosineAnnealingLR(opt_enc, cfg["epochs"], eta_min=LR * 0.01)
    loader = DataLoader(TensorDataset(Xt, yt, idx_t),
                        batch_size=cfg["batch"], shuffle=True, drop_last=True)
    best_score = np.inf; best_enc = best_head = None; no_imp = 0
    for epoch in range(cfg["epochs"]):
        alpha = GRL_MAX * max(0.1, epoch / max(1, cfg["epochs"]))
        adv.set_alpha(alpha); enc.train(); head.train(); adv.train()
        for xb, yb, bidx in loader:
            z_d = enc(xb).detach()
            for _ in range(ADV_STEPS):
                opt_adv.zero_grad(set_to_none=True)
                m_out, i_out = adv(z_d, alpha=0)
                la = (sum(nn.functional.binary_cross_entropy_with_logits(
                          m_out[:, i], sv_t[i][bidx]) for i in range(n_attrs)) / n_attrs
                      + ce(i_out, inter_t[bidx]))
                la.backward(); nn.utils.clip_grad_norm_(adv.parameters(), 1.0); opt_adv.step()
            opt_enc.zero_grad(set_to_none=True)
            z = enc(xb); logits = head(z); prob = torch.sigmoid(logits)
            task_loss = bce(logits, yb)
            fair_loss = torch.tensor(0.0, device=DEVICE)
            m_adv, i_adv = adv(z)
            for i, sv in enumerate(sv_t):
                tgt = sv[bidx]
                if metric == "eo":
                    sf = tgt.float(); yf = yb.float()
                    g0y1=(1-sf)*yf; g1y1=sf*yf; g0y0=(1-sf)*(1-yf); g1y0=sf*(1-yf)
                    if all(g.sum() > 1e-6 for g in [g0y1, g1y1, g0y0, g1y0]):
                        tpr0=(prob*g0y1).sum()/(g0y1.sum()+eps); tpr1=(prob*g1y1).sum()/(g1y1.sum()+eps)
                        fpr0=(prob*g0y0).sum()/(g0y0.sum()+eps); fpr1=(prob*g1y0).sum()/(g1y0.sum()+eps)
                        fair_loss += torch.max(torch.abs(tpr0-tpr1), torch.abs(fpr0-fpr1))
                else:
                    n0=(1-tgt).sum()+eps; n1=tgt.sum()+eps
                    fair_loss += torch.abs((prob*(1-tgt)).sum()/n0 - (prob*tgt).sum()/n1)
                fair_loss += nn.functional.binary_cross_entropy_with_logits(m_adv[:, i], tgt)
            fair_loss += ce(i_adv, inter_t[bidx])
            sg_loss = torch.tensor(0.0, device=DEVICE)
            if gamma_sg_static > 0:
                gm = prob.mean()
                for cls in range(n_inter):
                    cm = (inter_t[bidx] == cls)
                    if cm.sum() > 1:
                        sg_loss += torch.abs(prob[cm].mean() - gm)
                sg_loss = sg_loss / max(n_inter, 1)
            loss = (2.0 * task_loss + 0.3 * fair_loss / n_attrs + gamma_sg_static * sg_loss)
            loss.backward()
            nn.utils.clip_grad_norm_(list(enc.parameters()) + list(head.parameters()), 0.5)
            opt_enc.step()
        sched.step()
        enc.eval(); head.eval()
        with torch.no_grad():
            pv = torch.sigmoid(head(enc(Xv))).cpu().numpy()
        auc = safe_auc(d["y_val"], pv)
        V, _, _, _ = audit(epoch, d["y_val"], pv, d["sv_val"], d["attr_names"], metric=metric)
        score = V + 0.12 * (1 - auc if not np.isnan(auc) else 0)
        if score < best_score:
            best_score = score; best_enc = copy.deepcopy(enc.state_dict())
            best_head  = copy.deepcopy(head.state_dict()); no_imp = 0
        else:
            no_imp += 1
        if no_imp >= PATIENCE: break
    enc.load_state_dict(best_enc); head.load_state_dict(best_head)
    enc.eval(); head.eval()
    with torch.no_grad():
        proba = torch.sigmoid(head(enc(Xte))).cpu().numpy()
    tag = (f"Adv-{metric.upper()} (static)" if gamma_sg_static == 0
           else f"SubgroupPenalty-{metric.upper()}")
    return evaluate(proba, d["y_te"], d["sv_te"], d["attr_names"], tag=tag)


# ════════════════════════════════════════════════════════════════════════════
#  Data loaders
# ════════════════════════════════════════════════════════════════════════════

def load_adult():
    from ucimlrepo import fetch_ucirepo
    repo  = fetch_ucirepo(id=2)
    X_df  = repo.data.features.copy()
    y_ser = repo.data.targets.copy()
    y_raw = y_ser.iloc[:, 0].astype(str).str.strip().str.rstrip(".")
    y     = (y_raw == ">50K").astype(int).values
    race_Black = (X_df["race"].astype(str).str.strip() == "Black").astype(int).values
    race_White = (X_df["race"].astype(str).str.strip() == "White").astype(int).values
    sex_Female = (X_df["sex"].astype(str).str.strip() == "Female").astype(int).values
    age_old    = (pd.to_numeric(X_df["age"], errors="coerce").fillna(0) >= 45).astype(int).values
    X_df = X_df.drop(columns=["race","sex","age","fnlwgt","education-num"], errors="ignore")
    X_df = pd.get_dummies(X_df)
    X    = X_df.values.astype(np.float32)
    valid = ~np.isnan(X).any(axis=1)
    X, y  = X[valid], y[valid]
    race_Black=race_Black[valid]; race_White=race_White[valid]
    sex_Female=sex_Female[valid]; age_old=age_old[valid]
    tr, te = strat_split(X, y, [race_Black, sex_Female, age_old])
    sc = StandardScaler()
    X_tr = sc.fit_transform(X[tr]); X_te = sc.transform(X[te])
    attr_names = ["race_Black","race_White","sex_Female","age_old"]
    sv_tr = [race_Black[tr], race_White[tr], sex_Female[tr], age_old[tr]]
    sv_te = [race_Black[te], race_White[te], sex_Female[te], age_old[te]]
    tr2, val = strat_split(X_tr, y[tr], sv_tr)
    return dict(ds_key="adult",
        X_tr=X_tr[tr2], y_tr=y[tr][tr2], X_val=X_tr[val], y_val=y[tr][val],
        X_te=X_te, y_te=y[te],
        sv_tr=[ensure_binary(sv[tr2]) for sv in sv_tr],
        sv_val=[ensure_binary(sv[val]) for sv in sv_tr],
        sv_te=[ensure_binary(sv) for sv in sv_te],
        attr_names=attr_names)

def load_acs_income():
    from folktables import ACSDataSource, ACSIncome
    ds  = ACSDataSource(survey_year="2018", horizon="1-Year", survey="person")
    acs = ds.get_data(states=["CA"], download=True)
    features, label, group = ACSIncome.df_to_numpy(acs)
    row_mask = (acs["AGEP"].fillna(-1) >= 16)
    for col in ["PINCP","COW","WKHP"]:
        if col in acs.columns:
            if col == "WKHP":
                row_mask = row_mask & (pd.to_numeric(acs[col], errors="coerce").fillna(0) >= 1)
            else:
                row_mask = row_mask & acs[col].notna()
    if "ESR" in acs.columns:
        row_mask = row_mask & (~acs["ESR"].isin(["b", None]) & acs["ESR"].notna())
    acs_feature_cols = ["AGEP","COW","SCHL","MAR","OCCP","POBP","RELP","WKHP","SEX","RAC1P"]
    sex_idx  = acs_feature_cols.index("SEX"); race_idx = acs_feature_cols.index("RAC1P")
    age_idx  = acs_feature_cols.index("AGEP")
    sex_col  = features[:, sex_idx]; race_col = features[:, race_idx]; age_col = features[:, age_idx]
    rW=(race_col==1).astype(int); rB=(race_col==2).astype(int)
    rA=(race_col==6).astype(int); sex=(sex_col==2).astype(int); age_old=(age_col>=45).astype(int)
    valid    = ~np.isnan(label.astype(float))
    features = features[valid].astype(np.float32); label = label[valid].astype(int)
    rW=rW[valid]; rB=rB[valid]; rA=rA[valid]; sex=sex[valid]; age_old=age_old[valid]
    tr, te = strat_split(features, label, [rW, rB, sex, age_old])
    sc = StandardScaler()
    X_tr = sc.fit_transform(features[tr]); X_te = sc.transform(features[te])
    sv_tr = [rW[tr], rB[tr], rA[tr], sex[tr], age_old[tr]]
    sv_te = [rW[te], rB[te], rA[te], sex[te], age_old[te]]
    attr_names = ["race_White","race_Black","race_Asian","sex_Female","age_old"]
    tr2, val = strat_split(X_tr, label[tr], sv_tr)
    return dict(ds_key="acs_income",
        X_tr=X_tr[tr2], y_tr=label[tr][tr2], X_val=X_tr[val], y_val=label[tr][val],
        X_te=X_te, y_te=label[te],
        sv_tr=[ensure_binary(sv[tr2]) for sv in sv_tr],
        sv_val=[ensure_binary(sv[val]) for sv in sv_tr],
        sv_te=[ensure_binary(sv) for sv in sv_te],
        attr_names=attr_names)

def load_acs_employment():
    from folktables import ACSDataSource, ACSEmployment
    ds  = ACSDataSource(survey_year="2018", horizon="1-Year", survey="person")
    acs = ds.get_data(states=["CA"], download=True)
    features, label, _ = ACSEmployment.df_to_numpy(acs)
    acs_emp_cols = ["AGEP","SCHL","MAR","DIS","ESP","CIT","MIG","MIL","ANC",
                    "NATIVITY","DEAR","DEYE","DREM","SEX","RAC1P"]
    sex_idx  = acs_emp_cols.index("SEX"); race_idx = acs_emp_cols.index("RAC1P")
    age_idx  = acs_emp_cols.index("AGEP"); dis_idx  = acs_emp_cols.index("DIS")
    sex_col  = features[:, sex_idx]; race_col = features[:, race_idx]
    age_col  = features[:, age_idx]; dis_col  = features[:, dis_idx]
    RACE_MAP = {1:0, 2:1, 3:3, 4:2, 5:2, 6:3, 7:3, 8:3, 9:3}
    race = np.array([RACE_MAP.get(int(r), 3) for r in race_col])
    sex  = (sex_col == 2).astype(int)
    rW=(race==0).astype(int); rB=(race==1).astype(int); rO=(race==3).astype(int)
    age_old=(age_col>=45).astype(int); disabled=(dis_col==1).astype(int)
    valid    = ~np.isnan(label.astype(float))
    features = features[valid].astype(np.float32); label = label[valid].astype(int)
    rW=rW[valid]; rB=rB[valid]; rO=rO[valid]; sex=sex[valid]
    age_old=age_old[valid]; disabled=disabled[valid]
    tr, te = strat_split(features, label, [rW, rB, sex, age_old, disabled])
    sc = StandardScaler()
    X_tr = sc.fit_transform(features[tr]); X_te = sc.transform(features[te])
    sv_tr = [rW[tr], rB[tr], rO[tr], sex[tr], age_old[tr], disabled[tr]]
    sv_te = [rW[te], rB[te], rO[te], sex[te], age_old[te], disabled[te]]
    attr_names = ["race_White","race_Black","race_Other","sex_Female","age_old","disabled"]
    tr2, val = strat_split(X_tr, label[tr], sv_tr)
    return dict(ds_key="acs_employment",
        X_tr=X_tr[tr2], y_tr=label[tr][tr2], X_val=X_tr[val], y_val=label[tr][val],
        X_te=X_te, y_te=label[te],
        sv_tr=[ensure_binary(sv[tr2]) for sv in sv_tr],
        sv_val=[ensure_binary(sv[val]) for sv in sv_tr],
        sv_te=[ensure_binary(sv) for sv in sv_te],
        attr_names=attr_names)

def load_communities_crime():
    from ucimlrepo import fetch_ucirepo
    repo = fetch_ucirepo(id=183)
    X_df = repo.data.features.copy()
    y_df = repo.data.targets.copy()
    y_cont = pd.to_numeric(y_df.iloc[:, 0], errors="coerce").values
    valid  = ~np.isnan(y_cont)
    y_cont = y_cont[valid]; X_df = X_df[valid].reset_index(drop=True)
    y = (y_cont > np.median(y_cont)).astype(int)
    def find_col(df, pats):
        for p in pats:
            m = [c for c in df.columns if p.lower() in c.lower()]
            if m: return pd.to_numeric(df[m[0]], errors="coerce")
        return None
    col_b = find_col(X_df, ["racepctblack","pctblack"])
    col_i = find_col(X_df, ["medIncome","medincome"])
    def binarise(col, high=True):
        if col is None: return np.zeros(len(y), int)
        c = col.fillna(col.median()).values
        return (c > np.median(c)).astype(int) if high else (c < np.median(c)).astype(int)
    s_race = binarise(col_b, high=True); s_inc = binarise(col_i, high=False)
    X_num = X_df.apply(pd.to_numeric, errors="coerce")
    X_num = X_num.dropna(axis=1, thresh=int(0.7*len(X_num))).fillna(X_num.median())
    drop_cols = [c for c in X_num.columns if any(p.lower() in c.lower()
                 for p in ["racepct","racePct","medIncome","ViolentCrimes","PctUnemployed"])]
    X = X_num.drop(columns=drop_cols, errors="ignore").values.astype(np.float32)
    tr, te = strat_split(X, y, [s_race, s_inc])
    sc = StandardScaler()
    X_tr = sc.fit_transform(X[tr]); X_te = sc.transform(X[te])
    sv_tr = [s_race[tr], s_inc[tr]]; sv_te = [s_race[te], s_inc[te]]
    attr_names = ["racial_composition","socioeconomic"]
    tr2, val = strat_split(X_tr, y[tr], sv_tr)
    return dict(ds_key="communities_crime",
        X_tr=X_tr[tr2], y_tr=y[tr][tr2], X_val=X_tr[val], y_val=y[tr][val],
        X_te=X_te, y_te=y[te],
        sv_tr=[ensure_binary(sv[tr2]) for sv in sv_tr],
        sv_val=[ensure_binary(sv[val]) for sv in sv_tr],
        sv_te=[ensure_binary(sv) for sv in sv_te],
        attr_names=attr_names)

LOADERS = {
    "adult"            : load_adult,
    "acs_income"       : load_acs_income,
    "acs_employment"   : load_acs_employment,
    "communities_crime": load_communities_crime,
}


# ════════════════════════════════════════════════════════════════════════════
#  PHASE 1 — Pareto frontier sweep
# ════════════════════════════════════════════════════════════════════════════

print_section("PHASE 1 — Pareto frontier sweep  [WMW AUC proxy, seed=42]")

pareto_curves = {ds: {"eo": [], "dp": []} for ds in RUN_DATASETS}
t_pareto = time.time()

for ds_key in RUN_DATASETS:
    print_subsection(f"Dataset: {ds_key.upper()}")
    cfg = FULL_CFG[ds_key]
    d   = LOADERS[ds_key]()

    for metric in ["eo", "dp"]:
        bp_base  = dict(BEST_PARAMS[ds_key][metric])
        sm       = SCORE_MODE[ds_key][metric]
        fw       = FG_CKPT_WEIGHT[ds_key][metric]
        mce      = MIN_CKPT_EPOCH[ds_key][metric]
        print(f"\n  [{metric.upper()}] Pareto sweep ...")
        prev_wc = None
        for pw in PARETO_SWEEP_W:
            hp_sweep = dict(bp_base); hp_sweep["pareto_w"] = pw
            r = run_agad_with_hparams(d, cfg, metric, hp_sweep, seed=42,
                                       verbose=False, score_mode=sm,
                                       fg_ckpt_weight=fw, min_ckpt_epoch=mce)
            wc = r[f"wc_{metric}"]
            pareto_curves[ds_key][metric].append({
                "pareto_w": pw, "wc_fairness": wc,
                "fg_fairness": r[f"fg_{metric}"],
                "acc": r["acc"], "auc": r["auc"],
            })
            changed = "" if prev_wc is None else (
                "  ← MOVED" if abs(wc - prev_wc) > 5e-4 else "  (same)")
            print(f"    pareto_w={pw:.2f}  WC-{metric.upper()}={wc:.4f}  "
                  f"AUC={r['auc']:.4f}  Acc={r['acc']:.4f}{changed}")
            prev_wc = wc

    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

with open(f"{WORK_DIR}/agad_final_pareto_curves.json", "w") as f:
    json.dump(pareto_curves, f, indent=2)
print(f"\n  Pareto curves → {WORK_DIR}/agad_final_pareto_curves.json")
print(f"  Phase 1 time : {(time.time()-t_pareto)/60:.1f} min")


# ════════════════════════════════════════════════════════════════════════════
#  PHASE 2 — Final evaluation  (3 seeds)
# ════════════════════════════════════════════════════════════════════════════

print_section(f"PHASE 2 — Final evaluation  ({len(SEEDS_FINAL)} seeds: {SEEDS_FINAL})")

t_phase2 = time.time()
all_results = {}

for ds_key in RUN_DATASETS:
    print_subsection(f"Dataset: {ds_key.upper()}")
    cfg = FULL_CFG[ds_key]
    d   = LOADERS[ds_key]()
    print(f"  Train={len(d['y_tr'])}  Val={len(d['y_val'])}  Test={len(d['y_te'])}")

    ds_results = {}

    # ── Baselines ──
    for bname, bfn in [
        ("vanilla",  lambda s: run_vanilla(d, cfg, seed=s)),
        ("adv_eo",   lambda s: _adv_static(d, cfg, "eo", 0.0, seed=s)),
        ("adv_dp",   lambda s: _adv_static(d, cfg, "dp", 0.0, seed=s)),
        ("sgpen_eo", lambda s: _adv_static(d, cfg, "eo", SGPEN_GAMMA[ds_key], seed=s)),
        ("sgpen_dp", lambda s: _adv_static(d, cfg, "dp", SGPEN_GAMMA[ds_key], seed=s)),
    ]:
        print(f"\n  -- {bname} --")
        seed_results = []
        for s in SEEDS_FINAL:
            print(f"    seed {s} ...", end=" ", flush=True)
            r = bfn(s)
            seed_results.append(r)
            print(f"WC-EO={r['wc_eo']:.4f}  WC-DP={r['wc_dp']:.4f}  "
                  f"Acc={r['acc']:.4f}  AUC={r['auc']:.4f}")
        ds_results[bname] = aggregate_seeds(seed_results)

    # ── AGAD (tuned) ──
    for metric in ["eo", "dp"]:
        key  = f"agad_{metric}_tuned"
        hp   = BEST_PARAMS[ds_key][metric]
        sm   = SCORE_MODE[ds_key][metric]
        fw   = FG_CKPT_WEIGHT[ds_key][metric]
        mce  = MIN_CKPT_EPOCH[ds_key][metric]
        print(f"\n  -- AGAD-{metric.upper()} (tuned) --")
        print(f"     hparams       : {hp}")
        print(f"     score_mode    : {sm}")
        print(f"     fg_ckpt_wt    : {fw}")
        print(f"     min_ckpt_epoch: {mce}")
        seed_results = []
        for s in SEEDS_FINAL:
            print(f"    seed {s} ...")
            r = run_agad_with_hparams(d, cfg, metric, hp, seed=s,
                                       verbose=True, epoch_verbose=True,
                                       score_mode=sm, fg_ckpt_weight=fw,
                                       min_ckpt_epoch=mce)
            seed_results.append(r)
        ds_results[key] = aggregate_seeds(seed_results)

    all_results[ds_key] = ds_results

    # ── Per-dataset summary ──
    print(f"\n{'─'*108}")
    print(f"  {'Method':<20}  "
          f"{'WC-EO':>18}  {'WC-DP':>18}  "
          f"{'FG-EO':>18}  {'FG-DP':>18}  "
          f"{'Acc':>14}  {'AUC':>14}")
    print(f"  {'─'*104}")
    for name, r in ds_results.items():
        def fmt(k): return f"{r[k]:.4f}±{r[k+'_std']:.4f}"
        print(f"  {name:<20}  "
              f"{fmt('wc_eo'):>18}  {fmt('wc_dp'):>18}  "
              f"{fmt('fg_eo'):>18}  {fmt('fg_dp'):>18}  "
              f"{fmt('acc'):>14}  {fmt('auc'):>14}")

    print(f"\n  AGAD vs best baseline:")
    for metric in ["eo", "dp"]:
        best_base_wc  = min(ds_results[f"adv_{metric}"][f"wc_{metric}"],
                            ds_results[f"sgpen_{metric}"][f"wc_{metric}"])
        best_base_fg  = min(ds_results[f"adv_{metric}"][f"fg_{metric}"],
                            ds_results[f"sgpen_{metric}"][f"fg_{metric}"])
        best_base_acc = max(ds_results[f"adv_{metric}"]["acc"],
                            ds_results[f"sgpen_{metric}"]["acc"])
        key = f"agad_{metric}_tuned"
        r   = ds_results[key]
        dwc  = best_base_wc  - r[f"wc_{metric}"]
        dfg  = best_base_fg  - r[f"fg_{metric}"]
        dacc = r["acc"]      - best_base_acc
        acc_ok = r["acc"] >= VANILLA_ACC[ds_key] - ACC_TOLERANCE
        print(f"    agad_{metric}  "
              f"ΔWC-{metric.upper()}={dwc:+.5f}  "
              f"ΔFG-{metric.upper()}={dfg:+.5f}  "
              f"ΔAcc={dacc:+.5f}  "
              f"({'✓' if dwc>0 else '✗'}  {'✓' if dfg>0 else '✗'})  "
              f"acc_ok={'✓' if acc_ok else '✗'}")

    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

phase2_time = time.time() - t_phase2


# ════════════════════════════════════════════════════════════════════════════
#  PHASE 3 — Publication-quality graphs
# ════════════════════════════════════════════════════════════════════════════

print_section("PHASE 3 — Generating publication-quality graphs")

plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 10,
    "axes.titlesize": 11, "axes.labelsize": 10,
    "xtick.labelsize": 8, "ytick.labelsize": 8,
    "legend.fontsize": 8, "figure.dpi": 150,
    "axes.spines.top": False, "axes.spines.right": False,
})

def fig1_bar_fairness():
    fig, axes = plt.subplots(2, 4, figsize=(16, 7), sharey=False)
    fig.suptitle("Figure 1: Worst-Case Fairness Gaps Across Datasets and Methods\n"
                 "(lower is better; ★ = AGAD final)",
                 fontsize=13, fontweight="bold", y=1.01)
    for col, ds_key in enumerate(RUN_DATASETS):
        for row, metric in enumerate(["eo", "dp"]):
            ax = axes[row][col]; wc_key = f"wc_{metric}"
            means  = [all_results[ds_key][m][wc_key]          for m in METHOD_ORDER]
            stds   = [all_results[ds_key][m][wc_key + "_std"] for m in METHOD_ORDER]
            colors = [PALETTE[m] for m in METHOD_ORDER]
            x = np.arange(len(METHOD_ORDER))
            bars = ax.bar(x, means, yerr=stds, color=colors, alpha=0.85,
                          capsize=3, error_kw={"linewidth": 1.2})
            ax.set_xticks(x)
            ax.set_xticklabels([METHOD_LABELS[m] for m in METHOD_ORDER],
                               rotation=45, ha="right", fontsize=7)
            ax.set_title(f"{DS_LABELS[ds_key]}\nWC-{'EO' if metric=='eo' else 'DP'}", fontsize=9)
            ax.set_ylabel("Gap" if col == 0 else "", fontsize=9)
            ax.yaxis.grid(True, linestyle="--", alpha=0.4); ax.set_axisbelow(True)
            for i, m in enumerate(METHOD_ORDER):
                if "agad" in m:
                    bars[i].set_edgecolor("black"); bars[i].set_linewidth(1.5)
    plt.tight_layout()
    path = f"{PLOT_DIR}/fig1_wc_fairness_bars.png"
    plt.savefig(path, bbox_inches="tight", dpi=150); plt.close()
    print(f"  Saved → {path}")

def fig2_pareto_curves():
    fig, axes = plt.subplots(2, 4, figsize=(16, 7), sharey=False)
    fig.suptitle("Figure 2: Pareto Frontier — Worst-Case Fairness Gap vs AUC\n"
                 "(sweep pareto_w ∈ [0.00–0.30]; lower-left = ideal)",
                 fontsize=13, fontweight="bold", y=1.01)
    eo_patch = mpatches.Patch(color="#2dc653", label="AGAD-EO")
    dp_patch = mpatches.Patch(color="#1a7c34", label="AGAD-DP")
    for col, ds_key in enumerate(RUN_DATASETS):
        for row, metric in enumerate(["eo", "dp"]):
            ax   = axes[row][col]
            pts  = pareto_curves[ds_key][metric]
            aucs = [p["auc"]         for p in pts]
            wcs  = [p["wc_fairness"] for p in pts]
            pws  = [p["pareto_w"]    for p in pts]
            color = "#2dc653" if metric == "eo" else "#1a7c34"
            ax.plot(aucs, wcs, "-o", color=color, linewidth=2, markersize=5, zorder=3)
            for auc, wc, pw in zip(aucs, wcs, pws):
                ax.annotate(f"{pw:.2f}", (auc, wc), textcoords="offset points",
                            xytext=(4, 3), fontsize=6, color="#555555")
            for bname, bcol in [
                ("adv_eo" if metric=="eo" else "adv_dp", "#4e9af1"),
                ("sgpen_eo" if metric=="eo" else "sgpen_dp", "#f4a261")
            ]:
                r = all_results[ds_key][bname]
                ax.scatter(r["auc"], r[f"wc_{metric}"],
                           marker="D", s=50, color=bcol, zorder=4, label=METHOD_LABELS[bname])
            ax.scatter(all_results[ds_key]["vanilla"]["auc"],
                       all_results[ds_key]["vanilla"][f"wc_{metric}"],
                       marker="x", s=60, color="#6c757d", zorder=4,
                       linewidths=1.5, label="Vanilla NN")
            ax.set_title(f"{DS_LABELS[ds_key]}\nWC-{'EO' if metric=='eo' else 'DP'} vs AUC",
                         fontsize=9)
            ax.set_xlabel("AUC" if row == 1 else "", fontsize=8)
            ax.set_ylabel("WC-Fairness Gap" if col == 0 else "", fontsize=8)
            ax.yaxis.grid(True, linestyle="--", alpha=0.3); ax.set_axisbelow(True)
            if col == 3 and row == 0:
                ax.legend(fontsize=6, loc="upper right")
    fig.legend(handles=[eo_patch, dp_patch], loc="lower center", ncol=2,
               fontsize=9, frameon=False, title="AGAD sweep", title_fontsize=9)
    plt.tight_layout()
    path = f"{PLOT_DIR}/fig2_pareto_curves.png"
    plt.savefig(path, bbox_inches="tight", dpi=150); plt.close()
    print(f"  Saved → {path}")

def fig3_violin_stability():
    fig, axes = plt.subplots(2, 4, figsize=(16, 7), sharey=False)
    fig.suptitle("Figure 3: Seed Stability — WC-Fairness Gaps\n"
                 f"({len(SEEDS_FINAL)} seeds: {SEEDS_FINAL})",
                 fontsize=13, fontweight="bold", y=1.01)
    for col, ds_key in enumerate(RUN_DATASETS):
        for row, metric in enumerate(["eo", "dp"]):
            ax = axes[row][col]; wc_key = f"wc_{metric}_all"
            data   = [all_results[ds_key][m].get(wc_key, []) for m in METHOD_ORDER]
            data   = [d if d else [0.0] for d in data]
            colors = [PALETTE[m] for m in METHOD_ORDER]
            vp = ax.violinplot(data, positions=np.arange(len(METHOD_ORDER)),
                               showmeans=True, showmedians=False, widths=0.6)
            for body, c in zip(vp["bodies"], colors):
                body.set_facecolor(c); body.set_alpha(0.6)
            vp["cmeans"].set_color("black"); vp["cmeans"].set_linewidth(1.5)
            ax.set_xticks(np.arange(len(METHOD_ORDER)))
            ax.set_xticklabels([METHOD_LABELS[m] for m in METHOD_ORDER],
                               rotation=45, ha="right", fontsize=7)
            ax.set_title(f"{DS_LABELS[ds_key]}\nWC-{'EO' if metric=='eo' else 'DP'}", fontsize=9)
            ax.set_ylabel("Gap" if col == 0 else "", fontsize=9)
            ax.yaxis.grid(True, linestyle="--", alpha=0.4); ax.set_axisbelow(True)
    plt.tight_layout()
    path = f"{PLOT_DIR}/fig3_seed_stability_violins.png"
    plt.savefig(path, bbox_inches="tight", dpi=150); plt.close()
    print(f"  Saved → {path}")

def fig4_fg_bars():
    fig, axes = plt.subplots(2, 4, figsize=(16, 7), sharey=False)
    fig.suptitle("Figure 4: Fine-Grained (Top-K Subgroup) Fairness Gaps\n"
                 "(mean gap over K=5 worst subgroups; lower is better)",
                 fontsize=13, fontweight="bold", y=1.01)
    for col, ds_key in enumerate(RUN_DATASETS):
        for row, metric in enumerate(["eo", "dp"]):
            ax = axes[row][col]; fg_key = f"fg_{metric}"
            means  = [all_results[ds_key][m][fg_key]          for m in METHOD_ORDER]
            stds   = [all_results[ds_key][m][fg_key + "_std"] for m in METHOD_ORDER]
            colors = [PALETTE[m] for m in METHOD_ORDER]
            x = np.arange(len(METHOD_ORDER))
            bars = ax.bar(x, means, yerr=stds, color=colors, alpha=0.85,
                          capsize=3, error_kw={"linewidth": 1.2})
            ax.set_xticks(x)
            ax.set_xticklabels([METHOD_LABELS[m] for m in METHOD_ORDER],
                               rotation=45, ha="right", fontsize=7)
            ax.set_title(f"{DS_LABELS[ds_key]}\nFG-{'EO' if metric=='eo' else 'DP'}", fontsize=9)
            ax.set_ylabel("Gap" if col == 0 else "", fontsize=9)
            ax.yaxis.grid(True, linestyle="--", alpha=0.4); ax.set_axisbelow(True)
            for i, m in enumerate(METHOD_ORDER):
                if "agad" in m:
                    bars[i].set_edgecolor("black"); bars[i].set_linewidth(1.5)
    plt.tight_layout()
    path = f"{PLOT_DIR}/fig4_fg_fairness_bars.png"
    plt.savefig(path, bbox_inches="tight", dpi=150); plt.close()
    print(f"  Saved → {path}")

fig1_bar_fairness()
fig2_pareto_curves()
fig3_violin_stability()
fig4_fg_bars()


# ════════════════════════════════════════════════════════════════════════════
#  Global summary table
# ════════════════════════════════════════════════════════════════════════════

print_section("GLOBAL SUMMARY TABLE — all datasets × methods")

METRICS_SHOW = [
    ("wc_eo","WC-EO"), ("wc_dp","WC-DP"),
    ("fg_eo","FG-EO"), ("fg_dp","FG-DP"),
    ("acc","Acc"),     ("auc","AUC"),
]

header = f"  {'Dataset':<22}  {'Method':<20}"
for _, label in METRICS_SHOW:
    header += f"  {label:>16}"
print(header)
print("  " + "─" * (22 + 20 + 16 * len(METRICS_SHOW) + 4))

for ds_key in RUN_DATASETS:
    for i, m in enumerate(METHOD_ORDER):
        r        = all_results[ds_key][m]
        ds_label = DS_LABELS[ds_key] if i == 0 else ""
        row      = f"  {ds_label:<22}  {METHOD_LABELS[m]:<20}"
        for k, _ in METRICS_SHOW:
            cell = f"{r[k]:.4f}±{r[k+'_std']:.4f}"
            row += f"  {cell:>16}"
        print(row)
    print()


# ════════════════════════════════════════════════════════════════════════════
#  Save results
# ════════════════════════════════════════════════════════════════════════════

def clean(obj):
    if isinstance(obj, dict):          return {k: clean(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)): return [clean(v) for v in obj]
    if isinstance(obj, float) and (np.isnan(obj) or np.isinf(obj)): return None
    if isinstance(obj, (np.floating, np.integer)): return obj.item()
    if isinstance(obj, np.ndarray):   return obj.tolist()
    return obj

total_time = time.time() - t_pareto
payload = {
    "results"       : all_results,
    "best_params"   : BEST_PARAMS,
    "pareto_curves" : pareto_curves,
    "runtime": {
        "total_min"   : total_time / 60,
        "phase2_min"  : phase2_time / 60,
    },
    "seeds"         : SEEDS_FINAL,
    "fixes_applied" : {
        "fix1" : "Epoch-level EO accumulator (stable gradient signal)",
        "fix2" : "WMW differentiable AUC proxy",
        "fix3" : "Unified adv.forward(alpha=0) for adversary update",
        "fix4" : "ACS Income DP — updated params from grid search",
        "fix5" : "Communities & Crime DP — Optuna-tuned params (hardcoded)",
        "fix8a": "ACS Income EO — top_k_loss=6, re-tuned alpha/lambda0, pareto_w=0",
        "fix8b": "ACS Employment EO — min_ckpt_epoch=15 (epoch-0 checkpoint bug fixed),"
                 " alpha 20→12, lambda0 0.95→0.75, top_k_loss 1→3, score_mode=with_auc",
    },
}
out_path = f"{WORK_DIR}/agad_final_results.json"
with open(out_path, "w") as f:
    json.dump(clean(payload), f, indent=2)


# ════════════════════════════════════════════════════════════════════════════
#  Final checklist
# ════════════════════════════════════════════════════════════════════════════

print_section("RUNTIME SUMMARY")
print(f"  Phase 1 (Pareto sweep) : {(total_time - phase2_time)/60:.1f} min")
print(f"  Phase 2 (3-seed eval)  : {phase2_time/60:.1f} min")
print(f"  Total                  : {total_time/60:.1f} min")
print(f"\n  Results  → {out_path}")
print(f"  Pareto   → {WORK_DIR}/agad_final_pareto_curves.json")
print(f"  Plots    → {PLOT_DIR}/")

print()
print("=" * 70)
print("  PASS/FAIL CHECKLIST (AGAD vs best baseline, 3-seed means)")
print("=" * 70)

all_pass = True
for ds_key in RUN_DATASETS:
    for metric in ["eo", "dp"]:
        key   = f"agad_{metric}_tuned"
        r     = all_results[ds_key][key]
        best_wc  = min(all_results[ds_key][f"adv_{metric}"][f"wc_{metric}"],
                       all_results[ds_key][f"sgpen_{metric}"][f"wc_{metric}"])
        best_fg  = min(all_results[ds_key][f"adv_{metric}"][f"fg_{metric}"],
                       all_results[ds_key][f"sgpen_{metric}"][f"fg_{metric}"])
        dwc  = best_wc - r[f"wc_{metric}"]
        dfg  = best_fg - r[f"fg_{metric}"]
        acc_ok = r["acc"] >= VANILLA_ACC[ds_key] - ACC_TOLERANCE
        wc_ok  = dwc > 0
        fg_ok  = dfg > 0
        status = "✓ PASS" if (wc_ok and fg_ok and acc_ok) else "✗ FAIL"
        if not (wc_ok and fg_ok and acc_ok): all_pass = False
        print(f"  {ds_key:<22} {metric.upper()}  "
              f"ΔWC={dwc:+.5f}({'✓' if wc_ok else '✗'})  "
              f"ΔFG={dfg:+.5f}({'✓' if fg_ok else '✗'})  "
              f"acc_ok={'✓' if acc_ok else '✗'}  "
              f"[{status}]")

print()
print(f"  {'ALL CHECKS PASSED ✓' if all_pass else 'SOME CHECKS FAILED — review above'}")

print()
print("  Seed stability (WC-std across 3 seeds):")
for ds_key in RUN_DATASETS:
    for metric in ["eo", "dp"]:
        key    = f"agad_{metric}_tuned"
        r      = all_results[ds_key][key]
        wc_std = r.get(f"wc_{metric}_std", float("nan"))
        print(f"    {ds_key:<22} {metric.upper()}  "
              f"WC-std={wc_std:.5f}  "
              f"{'stable' if wc_std < 0.015 else 'variable'}")

  AGAD v4 — FINAL CONSOLIDATED SCRIPT
  Device : cuda
  Seeds  : [42, 123, 7]

  FIXES APPLIED:
    FIX 1 — Epoch-level EO accumulator
    FIX 2 — WMW differentiable AUC proxy
    FIX 3 — Unified adv.forward(alpha=0)
    FIX 4 — ACS Income DP: grid-search params
    FIX 5 — Communities & Crime DP: Optuna params (hardcoded)
    FIX 8a — ACS Income EO: top_k_loss=6, re-tuned alpha/lambda0
    FIX 8b — ACS Employment EO: min_ckpt_epoch=15 (epoch-0 bug)
             + alpha 20→12, lambda0 0.95→0.75, score_mode=with_auc

  PHASE 1 — Pareto frontier sweep  [WMW AUC proxy, seed=42]

----------------------------------------------------------------------
  Dataset: ADULT
----------------------------------------------------------------------

  [EO] Pareto sweep ...
    pareto_w=0.00  WC-EO=0.0522  AUC=0.8509  Acc=0.7937
    pareto_w=0.05  WC-EO=0.0528  AUC=0.8526  Acc=0.7967  ← MOVED
    pareto_w=0.10  WC-EO=0.0551  AUC=0.8543  Acc=0.7928  ← MOVED
    pareto_w=0.15  WC-EO=0.0577  AUC=0.8559  Ac